In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import pfc1

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)
started = time.time()

check = pfc1.check_checksums()
print("checksums.md5 on the bucket:", check["sums"])
print(f"{pfc1.ZIP_NAME} MD5 still {pfc1.ZIP_MD5}:", check["zip_matches_pin"])

unit_db = pfc1.load_unit_db()
sessions_df = pfc1.load_sessions_df()
fig_index = pfc1.load_figure_index()
S = pfc1.load_all(unit_db)
print(f"loaded {len(S)} sessions of CRCNS {pfc1.DATASET_ID} ({pfc1.DATASET_VERSION}, doi:{pfc1.DATASET_DOI}) "
      f"in {time.time() - started:.0f} s")

## Data check

Every file of the zip, the sessions, the units of the code's unit table and the trial tables. Each
check of the description against the values is stated under its output.

In [ ]:
files = pfc1.list_files()
print(f"{len(files)} file members in the zip (resource forks under __MACOSX/ left out), "
      f"{files['size'].sum() / 1e9:.2f} GB uncompressed")
misc = files[files.name.str.contains("miscellaneous")]
print(misc.to_string(index=False))
paths = pfc1.session_paths()
print("\nSessions per rat:", paths.groupby("rat").size().to_dict(), "total", len(paths))
print("Sessions without a spike file in klusters0/:", paths.index[paths.kkp.isna()].tolist())
print("Sessions with a second spike file outside klusters0/:", paths.index[paths.kkp_extra.notna()].tolist())
print("Sessions in the data set and in the code's sessions_df.csv:",
      len(set(paths.index) & set(sessions_df.index)), "of", len(paths), "and", len(sessions_df))

In [ ]:
# The two duplicated spike files: are they the same spikes?
for sess, row in paths[paths.kkp_extra.notna()].iterrows():
    a = pfc1.load_old_pickle(pfc1.read_member(row.kkp))
    b = pfc1.load_old_pickle(pfc1.read_member(row.kkp_extra))
    print(f"{sess}: klusters0 file {a.shape} columns {list(a.columns)}, extra file {b.shape} columns {list(b.columns)}")
    print("   units:", sorted(a.unit.unique().astype(int)) == sorted(b.unit.unique().astype(int)),
          "; groups in klusters0 file", sorted(a.group.unique().astype(int)), "extra file", sorted(b.group.unique().astype(int)))
    b = b.rename(columns={"spike_time": "time"})
    for col in ["group", "time", "unit"]:
        x, y = a[col].to_numpy(float), b[col].to_numpy(float)
        print(f"   {col}: equal {np.array_equal(x, y)}; equal after sorting {np.array_equal(np.sort(x), np.sort(y))}; "
              f"max abs difference after sorting {np.abs(np.sort(x) - np.sort(y)).max():.3g}")

The descriptor says each session folder holds one spike file in `klusters0/`, and that the folder is
empty when no units were recorded. Three sessions (CR20B_120613, CR21A_120502, CR21A_120506) have only a
`README.txt` there. Two CR21A sessions also hold a second copy of their spike file at the session
level; the output above says whether the copies hold the same spikes. Only the `klusters0/` file is
read. The descriptor lists `miscellaneous/catch_trial_task` with one file per rat, which is there.

In [ ]:
inc = unit_db[unit_db.include]
print("unit_db.csv rows:", len(unit_db), "; include True:", len(inc))
print(pd.crosstab(inc.ratname, inc.region, margins=True).to_string())
# region rule of the descriptor: CR12B and CR17B all A1; otherwise unit < 500 mPFC, > 500 A1
rule = np.where(inc.ratname.isin(["CR12B", "CR17B"]), "A1", np.where(inc.unum < 500, "PFC", "A1"))
print("Units whose unit_db region agrees with the descriptor's unit-number rule:",
      int((rule == inc.region).sum()), "of", len(inc))
print("Group column equals the first digit of the unit number:",
      bool((inc.group == inc.unum // 100).all()))
missing = [u for u, r in inc.iterrows() if int(r.unum) not in S[r.session_name]["spikes"]]
print("Included units without spikes in their session's file:", missing)
not_in_db = sum(len(set(S[s]["kkp_units"]) - set(unit_db[unit_db.session_name == s].unum.astype(int))) for s in S)
print("Clusters in spike files that unit_db does not list:", not_in_db,
      "; clusters in the files in total:", sum(len(S[s]["kkp_units"]) for s in S))

The unit table of the paper's code lists 220 units with include True, 121 mPFC and 99 A1, as the
paper reports. Every one is in its session's spike file. The region column agrees with the
descriptor's rule on unit numbers wherever the output says so.

In [ ]:
rows = []
for s, d in S.items():
    t = d["trials"]
    rows.append({"session": s, "rat": d["rat"], "trials": len(t), "munged": int(t.is_munged.sum()),
                 "nonrandom": int((t.nonrandom == 1).sum()),
                 "neg_times": int((t[["cpoke_start", "stim_onset", "cpoke_stop", "choice_made"]] < 0).any(axis=1).sum()),
                 "nan_onset": int(t.stim_onset.isna().sum()),
                 "hold_min_ms": 1000 * (t.stim_onset - t.cpoke_start).min(),
                 "hold_max_ms": 1000 * (t.stim_onset - t.cpoke_start).max(),
                 "units": len(d["spikes"]),
                 "spk_t_min": min((v.min() for v in d["spikes"].values()), default=np.nan),
                 "spk_t_max": max((v.max() for v in d["spikes"].values()), default=np.nan),
                 "last_onset": t.stim_onset.max()})
T = pd.DataFrame(rows).set_index("session")
print(T.round(2).to_string())
print("\nColumns of trial_timings joined to trials_info:", list(next(iter(S.values()))["trials"].columns))
ti0 = next(iter(S.values()))["trials"]
print("stim_name values:", sorted(ti0.stim_name.unique()))
print("outcome values over all sessions:", sorted(set().union(*[set(d['trials'].outcome) for d in S.values()])))
print("block values:", sorted(set().union(*[set(d['trials'].block) for d in S.values()])))
sn = pd.concat([d["trials"][["stim_name", "stim_number", "block", "go_or_nogo", "correct_side"]] for d in S.values()]).drop_duplicates()
print("\nstim_number, block, go_or_nogo and correct_side of each stim_name:")
print(sn.sort_values("stim_number").to_string(index=False))

Hold durations by rat: the code asserts 0 to 100 ms for CR12B, CR17B and YT6A and 250 to 350 ms for
the other three; the ranges above agree. Times are in seconds; in every session the spikes start
near 0 s and end after the last stimulus onset, so the spike files cover the trials. That is
consistent with one clock; the stimulus responses below are the check that a task contrast lines
up. No trial has a negative time. A few trials have no stimulus onset (column nan_onset) and are
dropped wherever onsets are used. Two points disagree with the descriptor: the stimulus column is
named `stim_name`, not `stimulus_name`, and the stimulus numbers do not follow the descriptor's
order (`le_lc_go` is 3, not 1; the mapping is printed above). Neither is used: stimuli are taken by
name.

In [ ]:
# Stimulus timing against spikes: the auditory-responsive units (audresp good, weak or sustained in
# unit_db.csv) should fire more in their response window after the trial table's stimulus onsets
# than in the 50 ms before; the same windows shifted to onset + 1.7 s (inside the intertrial
# interval) are the null. An offset between the clocks would remove the contrast.
psth_rows = []
aud = unit_db[unit_db.include & unit_db.audresp.isin(["good", "weak", "sustained"])]
for ul, u in aud.iterrows():
    d = S[u.session_name]; sp = d["spikes"][int(u.unum)]
    on = pfc1.analysis_trials(d["trials"]).stim_onset.dropna().to_numpy()
    w = u.audresp_t2 - u.audresp_t1
    row = {"ulabel": ul, "session": u.session_name, "region": u.region}
    for name, shift in [("onset", 0.0), ("onset+1.7s", 1.7), ("onset-0.5s", -0.5)]:
        o = on + shift
        ev = pfc1.count_in_windows(sp, o + u.audresp_t1, o + u.audresp_t2).mean() / w
        pre = pfc1.count_in_windows(sp, o - 0.05, o).mean() / 0.05
        row[name] = ev - pre
    psth_rows.append(row)
P = pd.DataFrame(psth_rows)
for reg, g in P.groupby("region"):
    for col in ["onset", "onset+1.7s", "onset-0.5s"]:
        print(f"{reg} (n = {len(g)}), window aligned to {col}: evoked minus pre-window rate, median "
              f"{g[col].median():.2f} Hz, units above 0: {(g[col] > 0).sum()}, Wilcoxon p = {pfc1.signedrank_r(g[col]):.2g}")
per_sess = P.groupby("session")["onset"].median()
print("Sessions with auditory-responsive units:", len(per_sess), "; with a positive median response:", int((per_sess > 0).sum()))
print("Sessions with included units but no auditory-responsive unit (clock not checked this way):",
      sorted(set(unit_db[unit_db.include].session_name) - set(per_sess.index)))

The responses at the trial table's onset times, against those at shifted times, are the check that
the spike files and trial tables share a clock; sessions without an auditory-responsive unit are
listed, and for them the rule encoding below (counts in the 50 or 250 ms before onset, which differ
between blocks) is the only task-locked check.

## unit-table-link

> The unit and session tables of the paper's code match the data set's spike files and session
> folders, and the matched units have the paper's counts per region.

Units are matched by session name and unit number, sessions by name (claims file, unit-table-link).

In [ ]:
inc = unit_db[unit_db.include]
found = sum(int(r.unum) in S[r.session_name]["spikes"] for _, r in inc.iterrows())
rule = np.where(inc.ratname.isin(["CR12B", "CR17B"]), "A1", np.where(inc.unum < 500, "PFC", "A1"))
sess_found = sum(s in S for s in sessions_df.index)
ntr = pd.Series({s: len(S[s]["trials"]) - int(S[s]["trials"].is_munged.sum()) for s in sessions_df.index if s in S})
print(f"Units with include True: {len(inc)}; found in their session's spike file: {found}")
print("Per region:", inc.region.value_counts().to_dict())
print(f"Region agrees with the descriptor's unit-number rule: {int((rule == inc.region).sum())} of {len(inc)}")
print(f"Sessions of sessions_df.csv among the data set's folders: {sess_found} of {len(sessions_df)}")
print(f"n_trials equals trial_timings rows less munged trials: {int((ntr == sessions_df.loc[ntr.index, 'n_trials']).sum())} of {len(ntr)}")
print("Sessions of unit_db (any include):", unit_db.session_name.nunique(), "; with include True units:", inc.session_name.nunique())

**Result: reproduced.** All 220 units with include True have spikes in their session's spike file,
121 in mPFC and 99 in A1, the region column agrees with the descriptor's rule for all 220, and all 55
sessions of `sessions_df.csv` are in the data set, each with `n_trials` equal to its trial_timings
rows less the munged trials.

## fig2a-above-chance

> Trained rats perform significantly above 50% correct in both localization and pitch
> discrimination blocks in the recording sessions, with a median of 698 trials per session; the
> best rats reach about 85% in both blocks. Sessions not above 50% in either block (4 of 55) were
> discarded.

Performance as `2-1-behavres/perf.py`: hits over hits, errors and wrong-port trials of blocks 2
and 4, nonrandom 0, munged trials dropped (the code loads trials with `load_trials_info`). Sessions as
`main00.py`: all 55 of `sessions_df.csv` (all marked include) less the catch-trial session
CR20B_120613. Binomial test of hits against 0.5 in each block, two-sided (claims M5), with the
one-sided test shown as a variant.

In [ ]:
SESSIONS_BEH = [s for s in sessions_df.index[sessions_df.include.astype(bool)] if s != "CR20B_120613_001_behaving"]
beh = []
for s in SESSIONS_BEH:
    d = S[s]
    raw = pfc1.load_raw_trials_info(s, d["rat"])
    t = pfc1.analysis_trials(d["trials"])
    t = t[t.nonrandom == 0]
    r = {"session": s, "rat": d["rat"],
         "n_performed": int((raw.outcome != "future_trial").sum()),
         "n_timings": len(d["trials"]),
         "n_clean": int(((~d["trials"].is_munged.astype(bool)) & (d["trials"].nonrandom == 0)).sum()),
         "n_sessions_df": sessions_df.loc[s, "n_trials"]}
    for b, name in [(2, "LB"), (4, "PB")]:
        x = t[t.block == b]
        h = int((x.outcome == "hit").sum()); n = int(x.outcome.isin(["hit", "error", "wrong_port"]).sum())
        r[f"{name}_perf"] = 100 * h / n
        r[f"{name}_n"] = n
        r[f"{name}_p"] = stats.binomtest(h, n, 0.5).pvalue
        r[f"{name}_p_greater"] = stats.binomtest(h, n, 0.5, alternative="greater").pvalue
    beh.append(r)
BEH = pd.DataFrame(beh).set_index("session")
print(BEH.round(4).to_string())

In [ ]:
fail = (BEH.LB_p >= 0.05) | (BEH.PB_p >= 0.05) | (BEH.LB_perf <= 50) | (BEH.PB_perf <= 50)
print(f"Sessions analysed: {len(BEH)}; failing the binomial test in either block: {int(fail.sum())}")
print("Largest p in either block:", f"{BEH[['LB_p', 'PB_p']].max().max():.3g}",
      "; lowest performance:", f"{BEH[['LB_perf', 'PB_perf']].min().min():.1f}%")
for col, label in [("n_performed", "(a) trials performed (outcome not future_trial)"),
                   ("n_clean", "(b) munged and nonrandom trials dropped"),
                   ("n_timings", "variant, rows of trial_timings"),
                   ("n_sessions_df", "variant, n_trials of sessions_df.csv")]:
    print(f"median trials per session, {label}: {BEH[col].median():.1f} "
          f"(expected 698 +/- 69.8; within: {abs(BEH[col].median() - 698) <= 69.8})")
for col in ["n_performed", "n_clean"]:
    allS = [s for s in sessions_df.index]
    print(f"  over all 55 sessions, {col}:",
          np.median([(pfc1.load_raw_trials_info(s, S[s]['rat']).outcome != 'future_trial').sum() if col == 'n_performed'
                     else ((~S[s]['trials'].is_munged.astype(bool)) & (S[s]['trials'].nonrandom == 0)).sum() for s in allS]))
per_rat = BEH.groupby("rat")[["LB_perf", "PB_perf"]].median()
per_rat["sessions"] = BEH.groupby("rat").size()
per_rat["both_within_85pm8.5"] = (abs(per_rat.LB_perf - 85) <= 8.5) & (abs(per_rat.PB_perf - 85) <= 8.5)
print("\nMedian performance over each rat's sessions (%):")
print(per_rat.round(1).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
order = ["CR12B", "CR17B", "YT6A", "CR20B", "CR21A", "CR24A"]
for i, rat in enumerate(order):
    g = BEH[BEH.rat == rat]
    ax.plot(np.full(len(g), 3 * i), g.LB_perf, "_", color="b", ms=12)
    ax.plot(np.full(len(g), 3 * i + 1), g.PB_perf, "_", color="r", ms=12)
ax.axhline(50, color="k", ls=":")
ax.set_xticks([3 * i + 0.5 for i in range(6)]); ax.set_xticklabels([f"Rat {pfc1.RAT_NUMBER[r]}" for r in order])
ax.set_ylim(40, 100); ax.set_ylabel("performance (%)")
ax.set_title("blue localization, red pitch discrimination")
plt.show()
print("Plotted range:", f"{BEH[['LB_perf', 'PB_perf']].min().min():.1f} to {BEH[['LB_perf', 'PB_perf']].max().max():.1f}%; none outside the axes")

**Result: reproduced.** None of the 54 analysed sessions fails the binomial test in either block,
so the criterion's bound of at most 4 failures holds (and reading (b), no failure, holds too). The
median trial count is within tolerance of 698 under reading (a), trials performed, and under
reading (b), after munged and nonrandom trials are dropped (printed above). CR20B, CR21A and CR24A
each have a median within 8.5 points of 85% in both blocks. The data set holds 55 sessions, all of
which pass, so the four discarded sessions of the paper are not in it.

## figs1a-go-better-than-nogo

> Performance is typically better on go trials than on nogo trials.

As `2-1-behavres/main02.py`: per rat, the mean over sessions of the performance on go and on nogo
trials of each block (perf.py, nonrandom 0, munged dropped), sessions as fig2a-above-chance.

In [ ]:
gng = []
for s in SESSIONS_BEH:
    d = S[s]; t = pfc1.analysis_trials(d["trials"]); t = t[t.nonrandom == 0]
    for b, name in [(2, "localization"), (4, "pitch discrimination")]:
        for g in ["go", "nogo"]:
            x = t[(t.block == b) & (t.go_or_nogo == g)]
            n = x.outcome.isin(["hit", "error", "wrong_port"]).sum()
            gng.append({"rat": d["rat"], "session": s, "block": name, "gng": g,
                        "perf": 100 * (x.outcome == "hit").sum() / n})
GNG = pd.DataFrame(gng)
tab = GNG.groupby(["rat", "block", "gng"]).perf.mean().unstack("gng")
tab["go_minus_nogo"] = tab.go - tab.nogo
print(tab.round(1).to_string())
print(f"\nPairs with go above nogo: {int((tab.go_minus_nogo > 0).sum())} of {len(tab)} (criterion: at least 7)")

**Result: reproduced.** Go performance exceeds nogo performance in the number of rat and block
pairs printed above, which meets the criterion's majority of at least 7 of 12.

## fig2d-block-switch

> After each block change rats switch rapidly to the new target; performance dips briefly
> during the cue trials at the start of a block and recovers within a few trials.

As `2-1-behavres/main04.py`: sessions as fig2a-above-chance, munged trials dropped, nonrandom 0,
trials 1 to 20 (the first block) dropped, trials folded on their position in the 160-trial cycle,
the hit fraction at each position pooled over sessions (the code's average weighted by trial
counts). The criterion's choice: the first trial of each block below the mean over that block's
non-cue trials (positions 21 to 80 or 101 to 160), and reaching that mean within the first 10
trials of the block (within 5 as a variant).

In [ ]:
fold = []
for s in SESSIONS_BEH:
    t = pfc1.analysis_trials(S[s]["trials"]); t = t[t.nonrandom == 0]
    t = t[t.index > 20]
    fold.append(pd.DataFrame({"pos": (t.index.to_numpy() - 1) % 160 + 1, "hit": (t.outcome == "hit").to_numpy()}))
FOLD = pd.concat(fold).groupby("pos").hit.agg(["mean", "size"])
for name, start in [("localization", 1), ("pitch discrimination", 81)]:
    block_mean = FOLD.loc[start + 20:start + 79, "mean"].mean()
    first = FOLD.loc[start, "mean"]
    early = FOLD.loc[start:start + 9, "mean"]
    reach10 = early.index[early >= block_mean]
    reach5 = FOLD.loc[start:start + 4, "mean"]
    print(f"{name}: hit fraction at block trial 1 = {first:.3f} (n = {FOLD.loc[start, 'size']}), "
          f"mean over non-cue trials = {block_mean:.3f}")
    print("   first 10 trials:", np.round(early.to_numpy(), 3).tolist())
    print(f"   first trial below the mean: {first < block_mean}; first position reaching the mean within 10: "
          f"{(reach10[0] - start + 1) if len(reach10) else None}; within 5: {bool((reach5 >= block_mean).any())}")
print("Trials per position, range:", FOLD["size"].min(), "to", FOLD["size"].max())
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(FOLD.index, 100 * FOLD["mean"], "k-")
for x0, c in [(0.5, "SkyBlue"), (80.5, "pink")]:
    ax.axvspan(x0, x0 + 20, color=c)
ax.set_xlabel("trial in the 160-trial cycle"); ax.set_ylabel("hits (%)"); ax.set_ylim(0, 100)
plt.show()
print("Plotted range:", f"{100 * FOLD['mean'].min():.1f} to {100 * FOLD['mean'].max():.1f}%")

**Result: reproduced.** In both blocks the hit fraction on the first trial is below the mean of
the block's non-cue trials and reaches that mean within the first 10 trials, at the positions
printed above; the within-5 variant is printed beside it.

## iti-median

> The median intertrial interval is 4.0 s (interquartile range 2.7 to 5.3 s).

The criterion's readings: (a) time between consecutive stimulus onsets, (b) time between
consecutive center-poke starts, each over all trials of the sessions with included units, pooled.
Consecutive rows of trial_timings are used; the same with munged trials dropped is a variant.

In [ ]:
iti_rows = []
SESS_UNITS = sorted(unit_db[unit_db.include].session_name.unique())
print("Sessions with included units:", len(SESS_UNITS))
res = {}
for label, col in [("(a) stimulus onsets", "stim_onset"), ("(b) center-poke starts", "cpoke_start")]:
    for drop in [False, True]:
        vals = []
        for s in SESS_UNITS:
            t = S[s]["trials"]
            if drop:
                t = pfc1.analysis_trials(t)
            vals.append(np.diff(t[col].dropna().to_numpy()))
        v = np.concatenate(vals)
        q1, q2, q3 = np.percentile(v, [25, 50, 75])
        ok = abs(q2 - 4.0) <= 0.4 and abs(q1 - 2.7) <= 0.27 and abs(q3 - 5.3) <= 0.53
        res[(label, drop)] = (q2, q1, q3)
        print(f"{label}{', munged dropped' if drop else ''}: median {q2:.2f} s, IQR {q1:.2f} to {q3:.2f} s "
              f"(n = {len(v)}; range {v.min():.2f} to {v.max():.1f} s); all within tolerance: {ok}")

**Result: reproduced.** Between consecutive stimulus onsets (reading (a)) the median interval is
3.99 s with an interquartile range of 2.65 to 5.23 s, all three within tolerance of 4.0, 2.7 and
5.3 s; reading (b), consecutive center-poke starts, gives the same values, and dropping munged
trials changes them by at most 0.01 s.

## Hold-period rule encoding

The units are the 220 with include True in `unit_db.csv` (claims M1). Trials are hits of blocks 2
and 4, nonrandom 0, munged dropped, holds of 50 ms or less dropped (M2). Spikes are counted in the
window ending at stimulus onset, 50 ms for CR12B, CR17B and YT6A and 250 ms for the others (M3). A
unit encodes the rule when the Mann-Whitney test of its counts in the two blocks (R's wilcox.test,
exact=FALSE) gives a Benjamini-Hochberg adjusted p below 0.05; reading (a) adjusts over all 220
units, as the code does, and reading (b) within each region. The preferred block is the one with the
higher mean count.

In [ ]:
H = pfc1.hold_table(S, unit_db)
print("Units:", len(H), H.region.value_counts().to_dict())
print("Hold-period spikes per unit, min:", H.n_spikes.min(), "(the Supplement's power rule asks for at least 20)")
print("Trials per unit, LB:", H.nLB.min(), "to", H.nLB.max(), "; PB:", H.nPB.min(), "to", H.nPB.max())
print("Hold rates (Hz), LB:", f"{H.LB_hz.min():.2f} to {H.LB_hz.max():.2f}", "; PB:", f"{H.PB_hz.min():.2f} to {H.PB_hz.max():.2f}")
enc_rows = []
for reading, col in [("(a) BH over all 220 units", "p_adj"), ("(b) BH within region", "p_adj_region")]:
    for reg in ["PFC", "A1"]:
        e = H[(H.region == reg) & (H[col] < 0.05)]
        nl, npb = int((e.pref == "LB").sum()), int((e.pref == "PB").sum())
        enc_rows.append({"reading": reading, "region": reg, "units": int((H.region == reg).sum()),
                         "encoding": len(e), "prefer_LB": nl, "prefer_PB": npb,
                         "binomial_p": stats.binomtest(nl, nl + npb, 0.5).pvalue})
ENC = pd.DataFrame(enc_rows)
print(ENC.round(4).to_string(index=False))
print("\nUnadjusted p < 0.05, for comparison:", H[H.p < 0.05].groupby("region").size().to_dict())
print("\nBy rat, reading (a):")
print(pd.crosstab([H.region, H.rat], H.p_adj < 0.05).rename(columns={True: "encoding", False: "not"}).to_string())

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for ax, reg in zip(axs, ["PFC", "A1"]):
    g = H[H.region == reg]
    lr = np.log10(g.PB_hz.clip(lower=0.1) / g.LB_hz.clip(lower=0.1))
    sig = g.p_adj < 0.05
    bins = np.linspace(-1.5, 1.5, 31)
    ax.hist([lr[sig], lr[~sig]], bins=bins, stacked=True, color=["k", "w"], edgecolor="k")
    ax.set_title(reg); ax.set_xlabel("log10 PB / LB hold rate (rates floored at 0.1 Hz)")
    print(reg, "log ratio range:", f"{lr.min():.2f} to {lr.max():.2f}", "; outside the axes:", int(((lr < -1.5) | (lr > 1.5)).sum()))
plt.show()

### fig3c-pfc-rule-encoding

> In the hold period of correct trials, 63% (76 of 121) of mPFC neurons individually encode the
> selection rule; 36 prefer localization and 40 pitch discrimination, and neither preference is
> significantly more common.

**Result: reproduced.** Under reading (a), the code's pooled adjustment, 76 of 121 mPFC units encode
the rule, 36 preferring localization and 40 pitch discrimination, the paper's counts exactly, and
the two-sided binomial test gives p = 0.73. Under reading (b) the counts are 81, 37 and 44, the
first outside the tolerance of 7.6, so the claim holds under reading (a) only.

### fig4c-a1-rule-encoding

> In the hold period of correct trials, 36% (36 of 99) of A1 neurons encode the selection rule; 13
> prefer localization and 23 pitch discrimination, and neither preference is significantly more
> common.

**Result: reproduced.** Under reading (a) 36 of 99 A1 units encode the rule, 13 preferring
localization and 23 pitch discrimination, the paper's counts, with binomial p = 0.13. Under reading
(b) the counts are 35, 12 and 23, also within tolerance (binomial p = 0.09).

### a1-rule-encoding-not-audresp-dependent

> In A1, rule encoding is not significantly more common among stimulus-responsive neurons (14 of
> 49) than among nonresponsive ones (22 of 50).

Responsiveness by M4 reading (a), audresp good, weak or sustained in `unit_db.csv`; rule encoding by
M3 reading (a); two-sided Fisher's exact test as `3-1-hold/main3.py`.

In [ ]:
H = H.join(unit_db[["audresp", "audresp_t1", "audresp_t2"]])
H["responsive"] = H.audresp.isin(["good", "weak", "sustained"])
a1 = H[H.region == "A1"]
# reading (b) of M4 is computed in the evoked-response section below and shown there
for col, label in [("p_adj", "reading (a)"), ("p_adj_region", "reading (b) of M3")]:
    tab = pd.crosstab(a1.responsive, a1[col] < 0.05)
    odds, p = stats.fisher_exact(tab.to_numpy())
    print(f"{label}: responsive {int(tab.loc[True, True])} of {int(tab.loc[True].sum())} encode the rule; "
          f"nonresponsive {int(tab.loc[False, True])} of {int(tab.loc[False].sum())}; Fisher's exact p = {p:.3f}, odds ratio {odds:.2f}")

**Result: reproduced.** 14 of 49 responsive and 22 of 50 nonresponsive A1 units encode the rule, the
paper's counts, and Fisher's exact test gives p above 0.05 (printed above), so the test finds no
difference.

### rule-encoding-effect-size

> Among rule-encoding neurons, the median increase in hold-period firing rate in the preferred
> block is 74.7% in mPFC and 99.7% in A1.

As `3-1-hold/main3.py`: per rule-encoding unit (reading (a)), the larger of the two ratios of mean
hold rates, minus 1; the median per region.

In [ ]:
ENCU = H[H.p_adj < 0.05].copy()
ENCU["increase_pct"] = 100 * (np.maximum(ENCU.mPB / ENCU.mLB, ENCU.mLB / ENCU.mPB) - 1)
for reg, exp in [("PFC", 74.7), ("A1", 99.7)]:
    v = ENCU[ENCU.region == reg].increase_pct
    print(f"{reg}: median increase {v.median():.1f}% over {len(v)} units (expected {exp} +/- {exp/10:.2f}); "
          f"range {v.min():.1f} to {v.max():.1f}%")
    print("   by rat:", ENCU[ENCU.region == reg].groupby("rat").increase_pct.median().round(1).to_dict())

**Result: reproduced.** The medians are 74.7% in mPFC and 99.8% in A1, within tolerance of the
paper's 74.7% and 99.7%.

### fig3ab-pfc-example-units

> Two simultaneously recorded mPFC units encode the rule with opposite preferences; one fires 12.1
> Hz in localization against 7.2 Hz in pitch discrimination (483 and 295 trials), the other 5.4 Hz
> in pitch discrimination against 2.7 Hz in localization, both p < 0.001.

The units are those `figures_generated_by_each_script.csv` names for panels 3A and 3B.

In [ ]:
print(fig_index[fig_index.panel.isin(["3A", "3B", "4A", "4B"])][["panel", "pyfile", "filename"]].to_string(index=False))
def example(ul):
    r = H.loc[ul]
    return {"unit": ul, "LB_hz": round(r.LB_hz, 2), "PB_hz": round(r.PB_hz, 2), "nLB": r.nLB, "nPB": r.nPB, "p_unadjusted": r.p}
ex3 = pd.DataFrame([example("CR24A_121021_001_behaving-257"), example("CR24A_121021_001_behaving-259")], index=["3A", "3B"])
print(ex3.to_string())

**Result: reproduced.** Unit 257 (3A) fires 12.07 Hz in localization against 7.19 Hz in pitch
discrimination over 483 and 295 trials, and unit 259 (3B) 5.41 Hz in pitch discrimination against
2.66 Hz in localization; both tests give p below 0.001. All values are within tolerance.

### fig4ab-a1-example-units

> Two simultaneously recorded A1 units encode the rule; one fires 8.0 Hz in localization against
> 4.8 Hz in pitch discrimination, the other 10.1 Hz in pitch discrimination against 2.0 Hz in
> localization, both p < 0.001.

The criterion names the units from the code's figure index, 4A unit 612 and 4B unit 712 of
CR20B_120602. The assignment the values give, 4A unit 712 and 4B unit 612, is shown as a variant (a
question to a reader asks which the claims file should use).

In [ ]:
ex4 = pd.DataFrame([example("CR20B_120602_001_behaving-612"), example("CR20B_120602_001_behaving-712")], index=["4A per index", "4B per index"])
print(ex4.to_string())
exp4 = {"A_LB": 8.0, "A_PB": 4.8, "B_PB": 10.1, "B_LB": 2.0}
def check4(ua, ub):
    a, b = H.loc[ua], H.loc[ub]
    got = {"A_LB": a.LB_hz, "A_PB": a.PB_hz, "B_PB": b.PB_hz, "B_LB": b.LB_hz}
    return {k: (round(v, 2), abs(v - exp4[k]) <= exp4[k] / 10) for k, v in got.items()}
print("\nAs the criterion names them (A = 612, B = 712):", check4("CR20B_120602_001_behaving-612", "CR20B_120602_001_behaving-712"))
print("Variant (A = 712, B = 612):", check4("CR20B_120602_001_behaving-712", "CR20B_120602_001_behaving-612"))
n = (H.loc["CR20B_120602_001_behaving-612", "nLB"], H.loc["CR20B_120602_001_behaving-612", "nPB"])
print(f"Trial counts: localization {n[0]}, pitch discrimination {n[1]}; 312 and 253 in the assignment localization = 312")

**Result: not reproduced.** With the units as the criterion names them, unit 612 (4A) fires 2.0 Hz
in localization and 10.07 Hz in pitch discrimination, and unit 712 (4B) 8.01 and 4.76 Hz, so none
of the four rates is within tolerance of the value given for its panel. With the two units swapped,
all four rates match (8.01, 4.76, 10.07 and 2.00 Hz), both p are below 0.001, and the trial counts
are 312 localization and 253 pitch discrimination, as the 4A legend gives them. The code's figure
index labels the two panels the other way from the legends.

## Error trials

As `3-2-errhold/main1.py` and `main2.py`: hold counts (M3 windows, holds over 50 ms, munged dropped,
nonrandom 0) on all trials of blocks 2 and 4 by stimulus and outcome. Hits; interference trials,
wrong-port responses to stimuli whose distractor cues a go (localization le_lo_lc and ri_lo_lc,
pitch discrimination le_hi_pc and le_lo_pc); go-on-nogo errors (localization ri_hi_lc and ri_lo_lc,
pitch discrimination le_hi_pc and ri_hi_pc). Rule-encoding units (reading (a)) with at least 3
trials in every category; rates in Hz relabelled by preferred block (P) and non-preferred block
(NP), normalized by subtracting the hit rate in the non-preferred block. Wilcoxon signed-rank tests
across units (M5).

In [ ]:
err_rows, err_n = [], []
for ul, u in ENCU.iterrows():
    d = S[u.session]; sp = d["spikes"][int(ul.split("-")[1])]
    cc = pfc1.category_counts(sp, d["trials"], d["rat"], pfc1.ERR_CATEGORIES)
    n = {k: len(v) for k, v in cc.items()}
    err_n.append({"ulabel": ul, "region": u.region, **n})
    if min(n.values()) < 3:
        continue
    P, N = u.pref, ("LB" if u.pref == "PB" else "PB")
    rec = {"ulabel": ul, "region": u.region, "rat": u.rat}
    for c in ["hit", "interference", "error"]:
        rec[c + "_P"] = cc[f"{c}_{P}"].mean() / u.win_s
        rec[c + "_NP"] = cc[f"{c}_{N}"].mean() / u.win_s
    err_rows.append(rec)
ERR = pd.DataFrame(err_rows).set_index("ulabel")
ERRN = pd.DataFrame(err_n).set_index("ulabel")
print("Rule-encoding units kept / dropped for fewer than 3 trials in a category:")
print(pd.DataFrame({"kept": ERR.groupby("region").size(), "encoding": ERRN.groupby("region").size()}).to_string())
NORM = ERR.copy()
cols = ["hit_NP", "interference_NP", "hit_P", "interference_P", "error_NP", "error_P"]
NORM[cols] = ERR[cols].sub(ERR.hit_NP, axis=0)
for reg in ["PFC", "A1"]:
    g = NORM[NORM.region == reg]
    print(f"\n{reg} (n = {len(g)}): mean normalized rate (Hz):", g[cols].mean().round(2).to_dict())
    for c1, c2 in [("hit_NP", "interference_NP"), ("hit_P", "interference_P"), ("interference_P", "interference_NP")]:
        print(f"   {c1} vs {c2}: mean {c2} minus {c1} = {(g[c2] - g[c1]).mean():.2f} Hz, "
              f"units with {c2} higher {int((g[c2] > g[c1]).sum())} of {len(g)}, signed-rank p = {pfc1.signedrank_r(g[c1], g[c2]):.4f}")
    print("   by rat, mean interference minus hit in NP and in P:",
          {r: (round((x.interference_NP - x.hit_NP).mean(), 2), round((x.interference_P - x.hit_P).mean(), 2), len(x))
           for r, x in g.groupby("rat")})

### fig3d-pfc-interference

> In mPFC rule-encoding neurons, rule encoding is weakened on interference trials; the
> hold-period response on interference trials is higher than on correct trials in the
> non-preferred block and lower in the preferred block (57 neurons).

**Result: reproduced.** 57 mPFC units have enough trials, the paper's count. On interference trials
the rate is higher than on hits in the non-preferred block and lower in the preferred block, each
with signed-rank p near 0.001 (printed above).

### fig4d-a1-interference

> In A1 rule-encoding neurons the rule encoding is reversed on interference trials; the
> hold-period response on interference trials is greater in the non-preferred block than in the
> preferred block (16 neurons).

**Result: reproduced.** 16 A1 units have enough trials, the paper's count, and the interference rate
is higher in the non-preferred than in the preferred block, signed-rank p = 0.0011, below 0.01.

## figs2c-s3c-hold-by-trial-type

> In rule-encoding neurons of both regions, hold-period activity does not differ between
> correct go, correct nogo and go-on-nogo error trials within a block, while interference trials
> differ from all other trial types.

As `3-2-errhold/main3.py`: the eight categories of the Figure S2C legend, units with at least 3
trials in each, rates relabelled by preferred block and normalized by the mean of the go and nogo
hit rates of the non-preferred block (the code's normalization; a constant per unit, so it does not
change the paired tests). Signed-rank tests between every pair of categories within each block,
Bonferroni over the 12 comparisons of a region.

In [ ]:
TT_ORDER = ["go", "nogo", "gonogo_err", "interf"]
tt_rows = []
for ul, u in ENCU.iterrows():
    d = S[u.session]; sp = d["spikes"][int(ul.split("-")[1])]
    cc = pfc1.category_counts(sp, d["trials"], d["rat"], pfc1.TYPE_CATEGORIES)
    if min(len(v) for v in cc.values()) < 3:
        continue
    P, N = u.pref, ("LB" if u.pref == "PB" else "PB")
    rec = {"ulabel": ul, "region": u.region}
    for c in TT_ORDER:
        rec[("P", c)] = cc[f"{c}_{P}"].mean() / u.win_s
        rec[("NP", c)] = cc[f"{c}_{N}"].mean() / u.win_s
    tt_rows.append(rec)
TT = pd.DataFrame(tt_rows).set_index("ulabel")
comp = []
for reg in ["PFC", "A1"]:
    g = TT[TT.region == reg]
    for blk in ["P", "NP"]:
        for i in range(4):
            for j in range(i + 1, 4):
                c1, c2 = TT_ORDER[i], TT_ORDER[j]
                x, y = g[(blk, c1)], g[(blk, c2)]
                comp.append({"region": reg, "block": blk, "c1": c1, "c2": c2, "n": len(g),
                             "mean_c2_minus_c1": (y - x).mean(), "p": pfc1.signedrank_r(x, y)})
COMP = pd.DataFrame(comp)
COMP["p_bonf"] = COMP.groupby("region").p.transform(lambda p: np.minimum(1, p * len(p)))
COMP["involves_interf"] = (COMP.c1 == "interf") | (COMP.c2 == "interf")
print(COMP.round(4).to_string(index=False))
for reg in ["PFC", "A1"]:
    g = COMP[COMP.region == reg]
    absent = (g[~g.involves_interf].p_bonf >= 0.05).all()
    present = (g[g.involves_interf].p_bonf < 0.05).all()
    print(f"{reg}: no non-interference comparison significant: {absent}; every interference comparison significant: {present} "
          f"({int((g[g.involves_interf].p_bonf < 0.05).sum())} of {int(g.involves_interf.sum())})")

**Result: reproduced.** In each region (57 mPFC and 16 A1 units) none of the six comparisons among
correct go, correct nogo and go-on-nogo errors has a Bonferroni-corrected p below 0.05 (all equal to
1 after correction), and all six comparisons of interference trials with another category have
corrected p below 0.05 (largest 0.035, A1 non-preferred block, go-on-nogo against interference).

## figs2f-s3f-bidirectional-vs-spont

> Across rule-encoding neurons in both regions, the hold-period rate is significantly above the
> spontaneous rate in the preferred block and significantly below it in the non-preferred block.

As `3-3-vs_spont/main2.py` and `main3.py`: the spontaneous rate of each unit is the spike count over
the time between its session's first and last stimulus onset lying more than 2 s from every onset
(all trials of trial_timings, munged included, time in 0.1 s steps). Hold rates as M3. Signed-rank
tests of hold rate against spontaneous rate, in each region for the localization-preferring and
pitch-discrimination-preferring units apart, each block. The pooled test per region is a variant.

In [ ]:
H["spont"] = [pfc1.spont_rate(S[r.session]["spikes"][int(ul.split("-")[1])], S[r.session]["trials"]) for ul, r in H.iterrows()]
print("Spontaneous rates (Hz):", f"{H.spont.min():.2f} to {H.spont.max():.2f}", "; NaN:", int(H.spont.isna().sum()))
sp_rows = []
E2 = H[H.p_adj < 0.05]
for reg in ["PFC", "A1"]:
    for pref in ["LB", "PB"]:
        g = E2[(E2.region == reg) & (E2.pref == pref)]
        np_ = "PB" if pref == "LB" else "LB"
        pr, nr = g[f"{pref}_hz"], g[f"{np_}_hz"]
        sp_rows.append({"region": reg, "pref": pref, "n": len(g),
                        "pref_minus_spont": (pr - g.spont).mean(), "p_pref": pfc1.signedrank_r(pr, g.spont),
                        "nonpref_minus_spont": (nr - g.spont).mean(), "p_nonpref": pfc1.signedrank_r(nr, g.spont)})
    g = E2[E2.region == reg]
    pr = np.where(g.pref == "PB", g.PB_hz, g.LB_hz); nr = np.where(g.pref == "PB", g.LB_hz, g.PB_hz)
    sp_rows.append({"region": reg, "pref": "pooled (variant)", "n": len(g),
                    "pref_minus_spont": (pr - g.spont).mean(), "p_pref": pfc1.signedrank_r(pr, g.spont),
                    "nonpref_minus_spont": (nr - g.spont).mean(), "p_nonpref": pfc1.signedrank_r(nr, g.spont)})
SP = pd.DataFrame(sp_rows)
SP["holds"] = (SP.pref_minus_spont > 0) & (SP.p_pref < 0.05) & (SP.nonpref_minus_spont < 0) & (SP.p_nonpref < 0.05)
print(SP.round(4).to_string(index=False))
print("Groups (of 4) in which both hold:", int(SP[~SP.pref.str.startswith("pooled")].holds.sum()))

**Result: reproduced.** In all four region and preference groups the preferred-block hold rate is
above the spontaneous rate and the non-preferred-block rate below it, each with signed-rank p below
0.05 (largest 0.025, A1 localization-preferring units against their non-preferred block, n = 13).
The pooled variant gives p of 0.0002 or less in both regions.

## figs2gh-s3gh-rt-mt

> Anticipatory firing rate does not correlate with reaction time or movement time; few neurons
> correlate individually and the mean correlation does not differ from zero.

As `3-4-RT_vs_hold/main.py` and `main3.py`: per unit (all 220) and block, the correlation
(linear regression) between the square root of the hold count and reaction time (center-port exit
minus stimulus onset) on all hits, and movement time (choice-port entry minus stimulus onset) on go
hits; munged dropped, nonrandom 0, no hold-length cut (the code counts on all trials). One-sample
t-test of the r values against 0 per region, block and measure; individually significant units by
Benjamini-Hochberg over each set. The variant takes movement time from center-port exit.

In [ ]:
rt_rows = []
for ul, u in H.iterrows():
    d = S[u.session]; sp = d["spikes"][int(ul.split("-")[1])]
    t = pfc1.analysis_trials(d["trials"]); t = t[(t.nonrandom == 0) & (t.outcome == "hit")]
    for b, bn in [(2, "LB"), (4, "PB")]:
        x = t[t.block == b]
        c = np.sqrt(pfc1.hold_counts(sp, x, d["rat"]))
        rt = (x.cpoke_stop - x.stim_onset).to_numpy()
        g = x.go_or_nogo.to_numpy() == "go"
        mt = (x.choice_made - x.stim_onset).to_numpy()
        mt2 = (x.choice_made - x.cpoke_stop).to_numpy()
        rec = {"ulabel": ul, "region": u.region, "block": bn}
        for name, xx, yy in [("RT", c, rt), ("MT", c[g], mt[g]), ("MT_from_exit", c[g], mt2[g])]:
            if np.std(xx) == 0:
                rec["r_" + name], rec["p_" + name] = np.nan, np.nan
            else:
                lr = stats.linregress(xx, yy)
                rec["r_" + name], rec["p_" + name] = lr.rvalue, lr.pvalue
        rt_rows.append(rec)
RT = pd.DataFrame(rt_rows)
print("Fits with a constant count (nothing to fit), dropped:", RT[["r_RT", "r_MT", "r_MT_from_exit"]].isna().sum().to_dict())
out = []
for name in ["RT", "MT", "MT_from_exit"]:
    for (reg, bn), g in RT.groupby(["region", "block"]):
        r = g["r_" + name].dropna(); p = g["p_" + name].dropna()
        out.append({"measure": name, "region": reg, "block": bn, "n": len(r), "mean_r": r.mean(),
                    "t_p": stats.ttest_1samp(r, 0).pvalue, "indiv_sig_BH": int((pfc1.bh(p) < 0.05).sum())})
RTO = pd.DataFrame(out)
print(RTO.round(4).to_string(index=False))
main = RTO[RTO.measure != "MT_from_exit"]
print("Main measures with mean r significantly different from 0:", int((main.t_p < 0.05).sum()), "of", len(main))

**Result: reproduced.** For none of the eight region, block and measure combinations does the mean
correlation differ from zero (one-sample t-test p from 0.052 to 0.90; the smallest, A1 localization
reaction time, has mean r = 0.016). Between 0 and 5 units of 99 or 121 correlate individually after
Benjamini-Hochberg. Movement time from center-port exit gives the same (p from 0.49 to 0.98).

## fig5ab-single-unit-onset

> The median rule-encoding unit first shows a significant block preference 0.55 s before the
> stimulus in A1 and 0.625 s before it in mPFC, so most develop it before the trial is initiated.

As `5-2-timecourse_exemplars/main0_dumpcounts.py` to `main4.py`: per rule-encoding unit, hits of
blocks 2 and 4 (nonrandom 0, munged dropped), spikes within the window around stimulus onset
smoothed per trial with a 50 ms Gaussian and sampled every 50 ms (the code's sample points
`linspace(-W, W, 2W/0.05, endpoint=False)` used as bin edges, values at the centres); a
Mann-Whitney test between blocks in each bin, p adjusted by Benjamini-Hochberg over all bins and
units together; the interval is the run of significant bins with the preferred direction reaching
back from the bin just before onset (`refbin`, centre -25 ms), and its first bin is the onset,
`(t1 - refbin) * 0.05` s. The median over units per region (`main4.py` takes all intervals). The
window is W = 3 s (Results; main reading) and W = 5 s (the code; variant).

In [ ]:
def single_unit_onsets(W, disc=True):
    edges = np.linspace(-W, W, int(round(2 * W / 0.05)), endpoint=False)
    centers = edges[:-1] + np.diff(edges) / 2
    refbin = int(np.argmin(np.abs(edges - 0.0))) - 1
    pv, au, labels = [], [], []
    for ul, u in ENCU.iterrows():
        d = S[u.session]; sp = d["spikes"][int(ul.split("-")[1])]
        t = pfc1.hit_trials(d["trials"], blocks=[2, 4])
        sm = {b: pfc1.smoothed_trials(sp, t[t.block == b].stim_onset.to_numpy(), centers, 0.05, lo=-W, hi=W, discretize=disc) for b in (2, 4)}
        res = stats.mannwhitneyu(sm[4], sm[2], axis=0, alternative="two-sided", method="asymptotic", use_continuity=True)
        p = np.where(np.isnan(res.pvalue), 1.0, res.pvalue)
        pv.append(p); au.append(res.statistic / (sm[4].shape[0] * sm[2].shape[0])); labels.append(ul)
    pv, au = np.array(pv), np.array(au)
    padj = pfc1.bh(pv.ravel()).reshape(pv.shape)
    rows = []
    for k, ul in enumerate(labels):
        t1, t2 = pfc1.first_bin_of_interval(padj[k], au[k], refbin)
        rows.append({"ulabel": ul, "region": ENCU.loc[ul, "region"], "rat": ENCU.loc[ul, "rat"],
                     "onset_s_before": -(t1 - refbin) * 0.05, "end_s_after": (t2 - refbin) * 0.05,
                     "refbin_sig": padj[k, refbin] <= 0.05})
    return pd.DataFrame(rows).set_index("ulabel"), centers[refbin]

t0 = time.time()
ONS = {}
for W, disc in [(3, True), (5, True), (3, False), (5, False)]:
    ONS[(W, disc)], refc = single_unit_onsets(W, disc)
    print(f"W = {W} s, {'spikes moved to the sample grid (the code)' if disc else 'exact spike times (variant)'} (refbin centre {refc * 1000:.0f} ms):")
    for reg, exp in [("A1", 0.55), ("PFC", 0.625)]:
        g = ONS[(W, disc)][ONS[(W, disc)].region == reg]
        q = np.percentile(g.onset_s_before, [25, 50, 75])
        print(f"   {reg}: median onset {q[1]:.3f} s before stimulus (expected {exp} +/- 0.05), IQR {q[0]:.3f} to {q[2]:.3f}, "
              f"n = {len(g)}, units whose refbin is significant {int(g.refbin_sig.sum())}; "
              f"onset earlier than 0.35 s before stimulus: {int((g.onset_s_before > 0.35).sum())}")
        print("      by rat (median):", g.groupby("rat").onset_s_before.median().round(3).to_dict())
print(f"({time.time() - t0:.0f} s)")

**Result: reproduced.** With the code's smoother (spikes moved to the 50 ms sample grid), the median
rule-encoding unit first shows its block preference 0.550 s before the stimulus in A1 (36 units)
and 0.625 s before it in mPFC (76 units), the paper's values, with W = 3 s and W = 5 s alike; both
exceed 0.35 s. With exact spike times (variant) the medians are 0.575 s (within tolerance) and
0.750 s (outside it), so the match depends on the code's discretization, which the paper does not
describe; the outcome follows the code, as reader step 2 directs where the Methods leave a choice
open.

## fig5c-population-onset

> The population activity of rule-encoding neurons in their preferred block ramps up before the
> stimulus and is significantly elevated earlier in mPFC (from 2.7 s before stimulus onset) than in
> A1 (from 0.88 s).

As `5-1-timecourse/main1.py`, `shared.py` and `main2b.py`: hits on the eight non-cue stimulus pairs,
nonrandom 0, munged dropped, locked to stimulus onset, spikes within 3 s; 50 ms Gaussian smoothing
sampled in 50 ms bins from -2.9 to 2.9 s; the rate of each stimulus pair is the smoothed count
over trials times bin width; each unit's rates z-scored over all bins and pairs, averaged within
the go and the nogo pairs of each block and then over go and nogo; the preferred block. Per bin a
one-sample t-test against 0 over units, Benjamini-Hochberg over bins; the onset is the centre of
the first significant bin with a positive mean.

In [ ]:
GROUPS = {("LB", "GO"): ["le_lo_lc", "le_hi_lc"], ("LB", "NO"): ["ri_lo_lc", "ri_hi_lc"],
          ("PB", "GO"): ["ri_lo_pc", "le_lo_pc"], ("PB", "NO"): ["ri_hi_pc", "le_hi_pc"]}

def zscored_group_rates(event, edges, smooth, disc=True):
    centers = edges[:-1] + np.diff(edges) / 2
    bw = np.mean(np.diff(edges))
    out = {}
    for ul, u in ENCU.iterrows():
        d = S[u.session]; sp = d["spikes"][int(ul.split("-")[1])]
        t = pfc1.hit_trials(d["trials"], stims=pfc1.MIXED_STIMNAMES)
        rates = {}
        for s in pfc1.MIXED_STIMNAMES:
            ev = t[t.stim_name == s][event].dropna().to_numpy()
            if smooth:
                sm = pfc1.smoothed_trials(sp, ev, centers, 0.05, lo=-3.0, hi=3.0, discretize=disc).sum(0)
            else:
                i0, i1 = np.searchsorted(sp, ev - 3.0), np.searchsorted(sp, ev + 3.0)
                rel = np.concatenate([sp[a:b] - e for a, b, e in zip(i0, i1, ev)]) if len(ev) else np.array([])
                sm = np.histogram(rel, bins=edges)[0]
            rates[s] = sm / (len(ev) * bw)
        R = pd.DataFrame(rates)
        z = (R - R.to_numpy().mean()) / R.to_numpy().std()
        out[ul] = {k: z[v].mean(axis=1).to_numpy() for k, v in GROUPS.items()}
    return out, centers

pop_rows = []
for disc in [True, False]:
  ZC, cent_c = zscored_group_rates("stim_onset", np.linspace(-2.9, 2.9, 117), smooth=True, disc=disc)
  for reg, exp in [("PFC", 2.7), ("A1", 0.88)]:
      uls = ENCU.index[ENCU.region == reg]
      arr = np.array([(ZC[u][(ENCU.loc[u, "pref"], "GO")] + ZC[u][(ENCU.loc[u, "pref"], "NO")]) / 2 for u in uls])
      p = stats.ttest_1samp(arr, 0, axis=0).pvalue
      padj = pfc1.bh(p)
      m = arr.mean(0)
      sig = np.where((padj < 0.05) & (m > 0))[0]
      first = cent_c[sig[0]] if len(sig) else np.nan
      lose = (sig[0] + np.where(~(padj[sig[0]:] < 0.05))[0][0]) if len(sig) and (~(padj[sig[0]:] < 0.05)).any() else None
      pop_rows.append({"smoothing": "code (grid)" if disc else "exact times (variant)", "region": reg, "n": len(uls), "first_sig_s_before": -first, "expected": exp,
                       "within": abs(-first - exp) <= exp / 10, "loses_sig_at_s": cent_c[lose] if lose is not None else np.nan,
                       "unadjusted_first_s_before": -cent_c[np.where((p < 0.05) & (m > 0))[0][0]]})
      print(f"{reg}: mean z at -2.9, -1, -0.025 s:", np.round(m[[0, np.argmin(abs(cent_c + 1)), np.argmin(abs(cent_c + 0.025))]], 3))
  if disc:
      ZC_code = ZC
POP = pd.DataFrame(pop_rows)
print(POP.round(3).to_string(index=False))
fig, ax = plt.subplots(figsize=(5, 3))
for reg, c in [("PFC", "purple"), ("A1", "orange")]:
    uls = ENCU.index[ENCU.region == reg]
    arr = np.array([(ZC_code[u][(ENCU.loc[u, "pref"], "GO")] + ZC_code[u][(ENCU.loc[u, "pref"], "NO")]) / 2 for u in uls])
    ax.plot(cent_c, arr.mean(0), color=c, label=reg)
    print(reg, "plotted mean z range:", f"{arr.mean(0).min():.2f} to {arr.mean(0).max():.2f}")
ax.axvline(0, color="k", lw=0.5); ax.set_xlabel("time from stimulus onset (s)"); ax.set_ylabel("z-scored rate, preferred block"); ax.legend()
plt.show()

**Result: reproduced.** The preferred-block population is first significantly above 0 at 2.725 s
before stimulus onset in mPFC (76 units) and 0.875 s before in A1 (36 units), within tolerance of
2.7 and 0.88 s, and mPFC is earlier. Smoothing with exact spike times gives the same bins.

## fig5d-go-nogo

> In the preferred block, after the rat leaves the center port, the population rate of
> rule-encoding neurons stays above baseline for several seconds on nogo trials and falls below
> baseline on go trials, in both mPFC and A1.

As `5-1-timecourse/main2.py`: locked to center-port exit, 50 ms bins from -3 to 3 s, counts without
smoothing, z-scored per unit as in fig5c-population-onset (baseline 0), the preferred block, go and
nogo apart. Readings: (a) per unit the mean z-scored rate from 0 to 3 s after exit, one-sample
t-test over units; (b) per-bin one-sample t-tests, Benjamini-Hochberg over bins, holding when nogo
is significantly above 0 and go significantly below 0 in most bins from 0.5 to 3 s.

In [ ]:
ZD, cent_d = zscored_group_rates("cpoke_stop", np.linspace(-3, 3, 121), smooth=False)
d_rows = []
for reg in ["PFC", "A1"]:
    uls = ENCU.index[ENCU.region == reg]
    for g in ["GO", "NO"]:
        arr = np.array([ZD[u][(ENCU.loc[u, "pref"], g)] for u in uls])
        post = (cent_d > 0) & (cent_d < 3)
        a = arr[:, post].mean(1)
        p_a = stats.ttest_1samp(a, 0).pvalue
        pb = pfc1.bh(stats.ttest_1samp(arr, 0, axis=0).pvalue)
        late = (cent_d > 0.5) & (cent_d < 3)
        m = arr.mean(0)
        frac = np.mean((pb[late] < 0.05) & ((m[late] > 0) if g == "NO" else (m[late] < 0)))
        d_rows.append({"region": reg, "trials": g, "n": len(uls), "(a) mean z 0-3 s": a.mean(), "(a) p": p_a,
                       "(b) fraction of 0.5-3 s bins significant in the claimed direction": frac})
D = pd.DataFrame(d_rows)
print(D.round(4).to_string(index=False))
for reg in ["PFC", "A1"]:
    g = D[D.region == reg].set_index("trials")
    a_ok = g.loc["NO", "(a) mean z 0-3 s"] > 0 and g.loc["NO", "(a) p"] < 0.05 and g.loc["GO", "(a) mean z 0-3 s"] < 0 and g.loc["GO", "(a) p"] < 0.05
    b_ok = (g["(b) fraction of 0.5-3 s bins significant in the claimed direction"] > 0.5).all()
    print(f"{reg}: reading (a) holds {a_ok}; reading (b) holds {b_ok}")

**Result: reproduced.** Under reading (a), in mPFC the mean z-scored rate over the 3 s after exit is
+0.437 on nogo trials (p < 0.0001) and -0.163 on go trials (p = 0.021); in A1 +0.264 (p = 0.0007)
and -0.421 (p = 0.0001). Reading (b) holds in mPFC (96% of nogo and 56% of go bins from 0.5 to 3 s
significant in the claimed direction) but not in A1, where only 28% of the nogo bins are.

## Evoked responses

Responsive units by M4 reading (a): audresp good, weak or sustained in `unit_db.csv`, with the
response window `audresp_t1` to `audresp_t2` (s after stimulus onset). Reading (b), the variant,
recomputes the window from the spikes as the Supplement's "Calculation of evoked responses" describes
(`pfc1.onset_window`): all trials of all stimuli (munged dropped), 1 ms Gaussian smoothing, each
0.5 ms bin from 0 to 50 ms after onset against all 0.5 ms bins of the 50 ms before (Mann-Whitney,
p < 0.05 uncorrected, the claims file's standard choice), the first run of bins all significantly
greater, windows under 1 ms discarded. The four atypical units the Supplement discards cannot be
identified, so none is discarded; windows that run to the end of the 50 ms (the code's class "lasts
to end") are shown with and without exclusion.
Evoked counts, as `6-1-evoked/main0.py`: spikes in the window on hits of every stimulus (the twelve
stimulus names), nonrandom 0, munged dropped.

In [ ]:
RESP = H[H.responsive].copy()
print("Responsive units:", RESP.groupby("region").size().to_dict())
print("Response windows (ms): t1", f"{1000 * RESP.audresp_t1.min():.1f} to {1000 * RESP.audresp_t1.max():.1f}",
      "; t2", f"{1000 * RESP.audresp_t2.min():.1f} to {1000 * RESP.audresp_t2.max():.1f}",
      "; length", f"{1000 * (RESP.audresp_t2 - RESP.audresp_t1).min():.1f} to {1000 * (RESP.audresp_t2 - RESP.audresp_t1).max():.1f}")
print("audresp classes of the included units:")
print(pd.crosstab(H.audresp, H.region).to_string())
EVK = {}
for ul, u in RESP.iterrows():
    d = S[u.session]; sp = d["spikes"][int(ul.split("-")[1])]
    t = pfc1.hit_trials(d["trials"], stims=pfc1.ALL_STIMNAMES)
    EVK[ul] = {s: pfc1.evoked_counts(sp, t[t.stim_name == s], u.audresp_t1, u.audresp_t2) for s in pfc1.ALL_STIMNAMES}
RESP["evok_dt"] = RESP.audresp_t2 - RESP.audresp_t1

In [ ]:
t0 = time.time()
win_b = {}
for ul, u in H.iterrows():
    d = S[u.session]
    on = pfc1.analysis_trials(d["trials"]).stim_onset.dropna().to_numpy()
    win_b[ul] = pfc1.onset_window(d["spikes"][int(ul.split("-")[1])], on)
H["b_t1"] = [w[0] if w else np.nan for w in win_b.values()]
H["b_t2"] = [w[1] if w else np.nan for w in win_b.values()]
H["b_len_ms"] = 1000 * (H.b_t2 - H.b_t1)
H["responsive_b"] = H.b_len_ms >= 1.0 - 1e-9
H["responsive_b_noend"] = H.responsive_b & (H.b_t2 < 0.05 - 1e-9)
print(f"Reading (b) computed for {len(H)} units in {time.time() - t0:.0f} s")
print(pd.crosstab([H.region, H.audresp], [H.responsive_b, H.responsive_b_noend], rownames=["region", "unit_db audresp"],
                  colnames=["(b) responsive", "(b) and not to end"]).to_string())
both = H[H.responsive & H.responsive_b]
print(f"Units responsive under both readings: {len(both)}; window start equal within 0.25 ms: {int((abs(both.b_t1 - both.audresp_t1) < 0.00025).sum())}, "
      f"end equal: {int((abs(both.b_t2 - both.audresp_t2) < 0.00025).sum())}")

### audresp-prevalence

> Auditory-responsive neurons are significantly rarer in mPFC (31 of 121) than in A1 (49 of 99).

Two-sided Fisher's exact test, as `6-2-evokmisc/main1.py`.

In [ ]:
tab = pd.crosstab(H.region, H.responsive)
table = [[int(tab.loc["A1", True]), int(tab.loc["A1", False])], [int(tab.loc["PFC", True]), int(tab.loc["PFC", False])]]
odds, p = stats.fisher_exact(table)
print(tab.to_string())
print(f"Reading (a): responsive mPFC {int(tab.loc['PFC', True])} of {int(tab.loc['PFC'].sum())}, A1 {int(tab.loc['A1', True])} of {int(tab.loc['A1'].sum())}; "
      f"Fisher's exact p = {p:.2g}, odds ratio {odds:.2f}")
for col in ["responsive_b", "responsive_b_noend"]:
    tb = pd.crosstab(H.region, H[col]).reindex(columns=[True, False], fill_value=0)
    o2, p2 = stats.fisher_exact([[int(tb.loc["A1", True]), int(tb.loc["A1", False])], [int(tb.loc["PFC", True]), int(tb.loc["PFC", False])]])
    print(f"Reading (b), {col}: mPFC {int(tb.loc['PFC', True])} of 121, A1 {int(tb.loc['A1', True])} of 99; Fisher's exact p = {p2:.2g}")

In [ ]:
# a1-rule-encoding-not-audresp-dependent under reading (b) of M4 (variant)
a1 = H[H.region == "A1"]
for col in ["responsive_b", "responsive_b_noend"]:
    tb = pd.crosstab(a1[col], a1.p_adj < 0.05).reindex(index=[True, False], columns=[True, False], fill_value=0)
    o3, p3 = stats.fisher_exact(tb.to_numpy())
    print(f"Variant, {col}: responsive {int(tb.loc[True, True])} of {int(tb.loc[True].sum())} encode the rule, nonresponsive "
          f"{int(tb.loc[False, True])} of {int(tb.loc[False].sum())}; Fisher's exact p = {p3:.3f}, odds ratio {o3:.2f}")

**Result: reproduced.** 31 of 121 mPFC and 49 of 99 A1 units are responsive, the paper's counts, with
Fisher's exact p below 0.001 (printed above). The counts come from the code's unit table, so this
reproduces the paper's tabulation, not the classification; the classification itself (reading (b))
was not recomputed.

### figs5cd-evoked-strength

> Onset responses are stronger in auditory-responsive A1 neurons than in mPFC neurons, 0.12
> against 0.02 extra spikes per trial (median) and 209% against 171% of baseline.

As `6-2-evokmisc/main0.py` and `main1.py`: all trials of every stimulus of each responsive unit
(munged dropped, no outcome or nonrandom filter, as the code's 'all' picker); baseline rate from the
50 ms before onset, evoked rate in the response window; extra spikes = (evoked minus baseline
rate) times the window length; percent = evoked over baseline rate. Mann-Whitney tests.

In [ ]:
s5 = []
for ul, u in RESP.iterrows():
    d = S[u.session]; sp = d["spikes"][int(ul.split("-")[1])]
    t = pfc1.analysis_trials(d["trials"]); t = t[t.stim_name.isin(pfc1.ALL_STIMNAMES) & t.stim_onset.notna()]
    on = t.stim_onset.to_numpy()
    pre = pfc1.count_in_windows(sp, on - 0.05, on).mean() / 0.05
    ev = pfc1.count_in_windows(sp, on + u.audresp_t1, on + u.audresp_t2).mean() / u.evok_dt
    s5.append({"ulabel": ul, "region": u.region, "rat": u.rat, "extra_spikes": (ev - pre) * u.evok_dt,
               "pct_baseline": 100 * ev / pre, "latency_ms": 1000 * (u.audresp_t1 + u.audresp_t2) / 2})
S5 = pd.DataFrame(s5).set_index("ulabel")
for col, exp in [("extra_spikes", {"A1": 0.12, "PFC": 0.02}), ("pct_baseline", {"A1": 209, "PFC": 171})]:
    a, b = S5[S5.region == "A1"][col], S5[S5.region == "PFC"][col]
    print(f"{col}: median A1 {a.median():.3f} (expected {exp['A1']}), mPFC {b.median():.3f} (expected {exp['PFC']}); "
          f"Mann-Whitney p = {pfc1.mannwhitney_r(a, b):.2g}; ranges A1 {a.min():.3f} to {a.max():.3f}, mPFC {b.min():.3f} to {b.max():.3f}")
for col in ["responsive_b", "responsive_b_noend"]:
    rows_b = []
    for ul, u in H[H[col]].iterrows():
        d = S[u.session]; sp = d["spikes"][int(ul.split("-")[1])]
        t = pfc1.analysis_trials(d["trials"]); on = t[t.stim_name.isin(pfc1.ALL_STIMNAMES)].stim_onset.dropna().to_numpy()
        w = u.b_t2 - u.b_t1
        pre = pfc1.count_in_windows(sp, on - 0.05, on).mean() / 0.05
        ev = pfc1.count_in_windows(sp, on + u.b_t1, on + u.b_t2).mean() / w
        rows_b.append({"region": u.region, "extra_spikes": (ev - pre) * w, "pct": 100 * ev / pre if pre > 0 else np.nan,
                       "latency_ms": 1000 * (u.b_t1 + u.b_t2) / 2})
    B5 = pd.DataFrame(rows_b)
    a_, b_ = B5[B5.region == "A1"], B5[B5.region == "PFC"]
    print(f"Variant, reading (b) {col}: extra spikes median A1 {a_.extra_spikes.median():.3f}, mPFC {b_.extra_spikes.median():.3f} "
          f"(p = {pfc1.mannwhitney_r(a_.extra_spikes, b_.extra_spikes):.2g}); percent A1 {a_.pct.median():.1f}, mPFC {b_.pct.median():.1f} "
          f"(p = {pfc1.mannwhitney_r(a_.pct.dropna(), b_.pct.dropna()):.2g}); latency A1 {a_.latency_ms.median():.2f}, mPFC {b_.latency_ms.median():.2f} ms "
          f"(p = {pfc1.mannwhitney_r(a_.latency_ms, b_.latency_ms):.2g})")
print("by rat (median extra spikes, pct), reading (a):")
print(S5.groupby(["region", "rat"])[["extra_spikes", "pct_baseline"]].median().round(3).to_string())

**Result: reproduced.** Over the 49 A1 and 31 mPFC responsive units the medians are 0.116 and
0.019 extra spikes per trial and 209.1% and 170.7% of baseline, all within tolerance of the paper's
0.12, 0.02, 209% and 171%, and the Mann-Whitney tests give p = 1.3e-09 and 0.00061, both below
0.001. Under reading (b) of M4 (variant) A1 is also stronger, with lower medians (0.064 and 0.005
extra spikes, 198% and 145%).

### figs5e-latency

> The median response latency of auditory-responsive mPFC neurons is slightly but significantly
> longer than in A1, 19.75 against 16.75 ms.

Latency is the centre of the response window (`6-2-evokmisc/main1.py`).

In [ ]:
a, b = S5[S5.region == "A1"].latency_ms, S5[S5.region == "PFC"].latency_ms
print(f"median latency mPFC {b.median():.2f} ms (expected 19.75 +/- 1.975), A1 {a.median():.2f} ms (expected 16.75 +/- 1.675); "
      f"Mann-Whitney p = {pfc1.mannwhitney_r(a, b):.3f}; ranges A1 {a.min():.2f} to {a.max():.2f}, mPFC {b.min():.2f} to {b.max():.2f} ms")

**Result: reproduced.** The median latency is 19.75 ms in mPFC and 16.75 ms in A1, the paper's
values, and the Mann-Whitney test gives p = 0.010. Under reading (b) of M4 (variant) the medians are
16.38 and 16.25 ms and do not differ (p = 0.41); the claim rests on the code's response windows.

### fig6cd-baseline-evoked-correlation

> Across auditory-responsive neurons, the change in hold-period rate between blocks correlates
> with the change in evoked rate, in A1 (43 neurons, trend slope close to 1) and in mPFC (17
> neurons), p < 0.001 in each.

As `6-3-evokgain/main0.py` and `main2.py`: units are dropped when a one-sample t-test of all their
evoked rates on non-cue hits against their hold rate (weighted by block trial counts) gives p > 0.05.
The evoked change is the stimulus-equalized bootstrap mean difference, pitch discrimination minus
localization (10000 draws, at least 5 trials per stimulus pair), the hold change the difference of
hold rates (M3). Linear regression of evoked change on hold change (scipy linregress). The A1 slope's
95% interval is the criterion's test of "close to 1".

In [ ]:
rng_gain = np.random.default_rng(6301)
NOBLOCK = ["le_hi", "ri_hi", "le_lo", "ri_lo"]
gain_rows, too_weak = [], []
for ul, u in RESP.iterrows():
    e = EVK[ul]
    allc = np.concatenate([e[s] for s in pfc1.MIXED_STIMNAMES]) / u.evok_dt
    nlb = sum(len(e[s + "_lc"]) for s in NOBLOCK); npb = sum(len(e[s + "_pc"]) for s in NOBLOCK)
    hold = (nlb * u.mLB + npb * u.mPB) / (nlb + npb) / u.win_s
    p_ev = stats.ttest_1samp(allc, hold).pvalue
    if p_ev > 0.05:
        too_weak.append(ul)
        continue
    rec = {"ulabel": ul, "region": u.region, "rat": u.rat, "hold_diff_hz": (u.mPB - u.mLB) / u.win_s, "p_hold_adj": u.p_adj}
    for sub in [False, True]:
        lb = [e[s + "_lc"] / u.evok_dt - (u.LB_hz if sub else 0) for s in NOBLOCK]
        pb = [e[s + "_pc"] / u.evok_dt - (u.PB_hz if sub else 0) for s in NOBLOCK]
        r = pfc1.stim_equalized_bootstrap(lb, pb, 10000, rng_gain)
        key = "subhold" if sub else "whold"
        rec[f"mdiff_{key}"] = np.nan if r is None else r["mdiff"]
        rec[f"p_{key}"] = np.nan if r is None else max(r["p_boot"], 1e-4)
    gain_rows.append(rec)
GAIN = pd.DataFrame(gain_rows).set_index("ulabel")
print("Too weak (dropped):", pd.Series([RESP.loc[u, "region"] for u in too_weak]).value_counts().to_dict(),
      "; kept:", GAIN.groupby("region").size().to_dict(), "; bootstrap not possible (a pair with < 5 trials):", int(GAIN.mdiff_whold.isna().sum()))
for reg, exp in [("A1", 43), ("PFC", 17)]:
    g = GAIN[(GAIN.region == reg) & GAIN.mdiff_whold.notna()]
    lr = stats.linregress(g.hold_diff_hz, g.mdiff_whold)
    tcrit = stats.t.ppf(0.975, len(g) - 2)
    print(f"{reg}: n = {len(g)} (expected {exp}); slope {lr.slope:.3f} (95% CI {lr.slope - tcrit * lr.stderr:.3f} to "
          f"{lr.slope + tcrit * lr.stderr:.3f}), r = {lr.rvalue:.3f}, p = {lr.pvalue:.2g}; hold change {g.hold_diff_hz.min():.1f} to "
          f"{g.hold_diff_hz.max():.1f} Hz, evoked change {g.mdiff_whold.min():.1f} to {g.mdiff_whold.max():.1f} Hz")

**Result: reproduced.** After the code's weak-response filter 43 A1 and 17 mPFC units remain, the
paper's counts. The regression of evoked change on hold change is positive in both, p = 0.0003 in
A1 and 1.7e-05 in mPFC, both below 0.001. The A1 slope is 0.98 with a 95% interval of 0.48 to 1.48,
which contains 1.

### evoked-beyond-baseline-rare

> After subtracting each block's baseline, only 6 of 43 A1 neurons (14%) change their evoked
> response significantly between blocks, 4 of the 6 from one rat (Rat 1), with higher rates in
> pitch discrimination.

As fig6cd-baseline-evoked-correlation with each block's hold rate subtracted from each trial's evoked
rate; bootstrap p floored at 1/10000, Benjamini-Hochberg within region (the code); the unadjusted
count is a variant.

In [ ]:
for reg in ["A1", "PFC"]:
    g = GAIN[(GAIN.region == reg) & GAIN.p_subhold.notna()].copy()
    g["p_adj"] = pfc1.bh(g.p_subhold)
    for label, col in [("BH within region", "p_adj"), ("unadjusted (variant)", "p_subhold")]:
        sig = g[g[col] < 0.05]
        print(f"{reg}, {label}: {len(sig)} of {len(g)} significant; by rat {sig.rat.value_counts().to_dict()}; "
              f"higher in pitch discrimination: {int((sig.mdiff_subhold > 0).sum())}; CR12B higher in PB: "
              f"{int(((sig.rat == 'CR12B') & (sig.mdiff_subhold > 0)).sum())} of {int((sig.rat == 'CR12B').sum())}")
        if reg == "A1" and col == "p_adj":
            print(sig[["rat", "mdiff_subhold", "p_subhold", "p_adj"]].round(4).to_string())

**Result: reproduced.** With the hold rate subtracted, 6 of 43 A1 units change their evoked response
significantly after Benjamini-Hochberg, 4 of them from CR12B (Rat 1), a majority, and all 4 CR12B
units are higher in pitch discrimination; the 2 others (CR17B) are higher in localization. The
unadjusted variant gives 12 of 43, 9 from CR12B. In mPFC 1 of 17 changes.

### fig6e-decoding

> Sound identity is decoded more accurately from A1 ensembles (21 ensembles, 49 neurons) than
> from mPFC ensembles (17 ensembles, 31 neurons), but for neither sound nor region is decoding
> better in one block than the other.

As `6-1-evoked/main1.py`, `main2.py`, `main4.py` and `decoders.py`: an ensemble is the responsive
units of one region in one session. Per ensemble, block and sound (noise burst side sL, warble
pitch sP), a logistic regression (scikit-learn, class weights balanced, C = 1; the liblinear solver,
the default of the code's scikit-learn) on the trial-by-trial evoked counts of hits of the four
stimulus pairs of the block, scored in sample as the fraction correct averaged over the two classes.
A type II three-way ANOVA of the scores on region, sound and block; paired t-tests between blocks
per region and sound, Benjamini-Hochberg over the four. Leave-one-out scores are a variant.

In [ ]:
from sklearn.linear_model import LogisticRegression
import statsmodels.formula.api as smf
import statsmodels.api as sm

def cateq_score(X, y, cv=False):
    X = np.asarray(X, float); y = np.asarray(y)
    if X.ndim == 1:
        X = X[:, None]
    def fit(Xt, yt):
        return LogisticRegression(class_weight="balanced", C=1.0, solver="liblinear").fit(Xt, yt)
    if cv:
        pred = np.empty(len(y), dtype=y.dtype)
        for i in range(len(y)):
            m = np.ones(len(y), bool); m[i] = False
            pred[i] = fit(X[m], y[m]).predict(X[i:i + 1])[0]
    else:
        pred = fit(X, y).predict(X)
    return float(np.mean([np.mean(pred[y == v] == v) for v in np.unique(y)]))

dec_rows = []
for (reg, sess), g in RESP.groupby(["region", "session"]):
    for block in ["lc", "pc"]:
        stims = [s for s in pfc1.MIXED_STIMNAMES if s.endswith(block)]
        X = np.column_stack([np.concatenate([EVK[ul][s] for s in stims]) for ul in g.index])
        sL = np.concatenate([np.full(len(EVK[g.index[0]][s]), s[:2] == "ri") for s in stims]).astype(int)
        sP = np.concatenate([np.full(len(EVK[g.index[0]][s]), s[3:5] == "hi") for s in stims]).astype(int)
        for var, y in [("sL", sL), ("sP", sP)]:
            dec_rows.append({"ensem": reg, "session": sess, "block": block, "var": var, "n_units": len(g), "n_trials": len(y),
                             "score": cateq_score(X, y), "score_cv": cateq_score(X, y, cv=True)})
DEC = pd.DataFrame(dec_rows)
nens = DEC.groupby("ensem").session.nunique(); nun = DEC.drop_duplicates(["ensem", "session"]).groupby("ensem").n_units.sum()
print("Ensembles:", nens.to_dict(), "; units in them:", nun.to_dict())
for col in ["score", "score_cv"]:
    fit = smf.ols(f"{col} ~ C(ensem) * C(var) * C(block)", data=DEC).fit()
    aov = sm.stats.anova_lm(fit, typ=2)
    print(f"\n{col}: type II ANOVA")
    print(aov.round(4).to_string())
    print("mean score by region:", DEC.groupby("ensem")[col].mean().round(3).to_dict())
    pv = []
    for reg in ["A1", "PFC"]:
        for var in ["sL", "sP"]:
            piv = DEC[(DEC.ensem == reg) & (DEC["var"] == var)].pivot(index="session", columns="block", values=col)
            pv.append({"ensem": reg, "var": var, "n": len(piv), "mean_lc": piv.lc.mean(), "mean_pc": piv.pc.mean(),
                       "p": stats.ttest_rel(piv.lc, piv.pc).pvalue})
    PV = pd.DataFrame(pv); PV["p_adj"] = pfc1.bh(PV.p)
    print(PV.round(4).to_string(index=False))

**Result: reproduced.** There are 21 A1 ensembles of 49 units and 17 mPFC ensembles of 31 units, the
paper's counts. The main effect of region has F = 39.3, p < 0.001, with A1 higher (mean score 0.570
against 0.521), and none of the four paired t-tests between blocks is significant (adjusted p 0.35
to 0.59; unadjusted 0.088 to 0.59), so both the region claim and the block claim (an absence) hold.
Leave-one-out scores give the same (region F = 21.6, block comparisons adjusted p 0.57 to 0.83).

### figs5fg-tuning-vs-rule

> There is no correlation between a neuron's change in anticipatory activity and its tuning for
> the noise burst or the warble, in A1 or mPFC.

As `6-1-evoked/main2.py` and `main6.py`: tuning is the single-unit in-sample decoding score (as
fig6e-decoding) of the cue sound on cue-trial hits of each responsive unit, the noise burst from
localization cue trials (le_lc_go, ri_lc_no) and the warble from pitch discrimination cue trials
(lo_pc_go, hi_pc_no); units with at least 10 evoked spikes on the cue trials and at least 25 cue
trials in each block; the anticipatory change is log10 of the pitch discrimination over localization
hold rate. Linear regression of the change on each tuning, per region.

In [ ]:
tun = []
for ul, u in RESP.iterrows():
    e = EVK[ul]
    rec = {"ulabel": ul, "region": u.region, "lograt": np.log10(u.mPB / u.mLB)}
    n_sp, n_tr = 0, []
    for block, (s0, s1) in [("lc", ("le_lc_go", "ri_lc_no")), ("pc", ("lo_pc_go", "hi_pc_no"))]:
        X = np.concatenate([e[s0], e[s1]]); y = np.r_[np.zeros(len(e[s0])), np.ones(len(e[s1]))].astype(int)
        rec[block + "_tuneQ"] = cateq_score(X, y) if len(np.unique(y)) == 2 else np.nan
        n_sp += X.sum(); n_tr.append(len(y))
    rec["n_spikes"], rec["min_trials"] = n_sp, min(n_tr)
    tun.append(rec)
TUN = pd.DataFrame(tun).set_index("ulabel")
keep = (TUN.n_spikes >= 10) & (TUN.min_trials >= 25)
print("Units kept:", TUN[keep].groupby("region").size().to_dict(), "; dropped:", TUN[~keep].groupby("region").size().to_dict())
for reg in ["A1", "PFC"]:
    g = TUN[keep & (TUN.region == reg)].dropna()
    for m in ["lc_tuneQ", "pc_tuneQ"]:
        lr = stats.linregress(g[m], g.lograt)
        print(f"{reg}, {m}: n = {len(g)}, r = {lr.rvalue:.3f}, p = {lr.pvalue:.3f}; tuning {g[m].min():.2f} to {g[m].max():.2f}")

**Result: reproduced.** After the code's spike and trial minimums (23 A1 and 9 mPFC units kept), none
of the four regressions of the anticipatory change on tuning is significant (r from -0.21 to 0.51,
p from 0.16 to 0.87).

### figs5hi-stimulus-pairs

> No stimulus pair shows a greater change in evoked response between blocks than the others, with
> or without baseline subtraction, and in absolute value; and no stimulus pair's change differs
> from zero.

As `6-1-evoked/main5.py`: per responsive unit and stimulus pair, the square root of the mean evoked
rate on hits in pitch discrimination minus that in localization; baseline-subtracted as the square
root of the evoked rate minus the square root of the block's hold rate. Kruskal-Wallis across the
four pairs per region, raw and baseline-subtracted, and on absolute values; Wilcoxon signed-rank
(scipy's default, as the code) against zero per pair, Benjamini-Hochberg over the four pairs.

In [ ]:
hi_rows = []
for ul, u in RESP.iterrows():
    e = EVK[ul]
    rec = {"ulabel": ul, "region": u.region}
    for s in NOBLOCK:
        lb = np.sqrt(e[s + "_lc"].mean() / u.evok_dt); pb = np.sqrt(e[s + "_pc"].mean() / u.evok_dt)
        rec[("raw", s)] = pb - lb
        rec[("subhold", s)] = (pb - np.sqrt(u.PB_hz)) - (lb - np.sqrt(u.LB_hz))
    hi_rows.append(rec)
HI = pd.DataFrame(hi_rows).set_index("ulabel")
tests = []
for reg in ["A1", "PFC"]:
    g = HI[HI.region == reg]
    for kind in ["raw", "subhold"]:
        for take_abs in [False, True]:
            vals = [np.abs(g[(kind, s)]) if take_abs else g[(kind, s)] for s in NOBLOCK]
            tests.append({"region": reg, "kind": kind, "abs": take_abs, "test": "Kruskal-Wallis", "pair": "all",
                          "p": stats.kruskal(*vals).pvalue, "n": len(g)})
            if not take_abs:
                ps = [stats.wilcoxon(v).pvalue for v in vals]
                for s, p, pa, v in zip(NOBLOCK, ps, pfc1.bh(ps), vals):
                    tests.append({"region": reg, "kind": kind, "abs": False, "test": "Wilcoxon vs 0 (BH)", "pair": s,
                                  "p": pa, "n": len(g), "median": float(np.median(v))})
TESTS = pd.DataFrame(tests)
print(TESTS.round(4).to_string(index=False))
print("Tests with p < 0.05:", int((TESTS.p < 0.05).sum()), "of", len(TESTS))

**Result: reproduced.** None of the 24 tests gives p below 0.05: the Kruskal-Wallis tests across
stimulus pairs give p from 0.061 to 0.92, raw, baseline-subtracted and absolute, and the Wilcoxon
tests of each pair against zero adjusted p from 0.12 to 0.99.

## Head angle

### figs2i-head-angle

> The rat's head angle during the center poke differs between blocks by about 30 degrees, 26.9,
> 36.6 and 24.1 degrees in the three scored sessions.

As `3-5-vidtrack/main0.py` and `main1.py`: the head angle of each scored trial (degrees), joined to
the trials of the hold analysis (M2: hits of blocks 2 and 4 with a hold over 50 ms); the difference
of the block means per session, compared with the three reported values as a sorted set.

In [ ]:
ANG = {s: pfc1.load_head_angles(s) for s in pfc1.VIDTRACK_SESSIONS}
ang_rows = []
for s, a in ANG.items():
    d = S[s]
    lb = pfc1.hold_trials(d["trials"], d["rat"], block=2, outcome="hit").index
    pb = pfc1.hold_trials(d["trials"], d["rat"], block=4, outcome="hit").index
    j = a.join(pd.Series(np.where(a.index.isin(lb), "LB", np.where(a.index.isin(pb), "PB", None)), index=a.index, name="block"))
    j = j.dropna(subset=["block", "angl"])
    ang_rows.append({"session": s, "scored_trials": len(a), "in_hold_analysis": len(j), "LB_mean": j[j.block == "LB"].angl.mean(),
                     "PB_mean": j[j.block == "PB"].angl.mean(), "angle_range": f"{a.angl.min():.1f} to {a.angl.max():.1f}"})
    ANG[s] = j
ANGT = pd.DataFrame(ang_rows).set_index("session")
ANGT["diff_LB_minus_PB"] = ANGT.LB_mean - ANGT.PB_mean
print(ANGT.round(2).to_string())
got = np.sort(np.abs(ANGT.diff_LB_minus_PB.to_numpy())); exp = np.sort([26.9, 36.6, 24.1])
print("sorted |differences|:", np.round(got, 2), "expected", exp, "within 10% each:", (np.abs(got - exp) <= exp / 10).tolist())

**Result: reproduced.** The localization minus pitch discrimination differences of mean head angle
are -26.92, -36.60 and -24.07 degrees in CR24A_121022, CR20B_120602 and CR17B_110803; their sizes,
sorted, are within tolerance of 24.1, 26.9 and 36.6 degrees. The paper's order of values matches
this order of sessions.

### figs2kl-s3kl-fev

> For most rule-encoding neurons, block explains far more of the hold-period variance than head
> angle; the median fraction explained by block is 94.1% in mPFC and 83.0% in A1 (head angle 5.9%
> and 17.0%), and only 3 of 16 mPFC and 1 A1 neuron are mostly explained by head angle.

As `3-5-vidtrack/main7.py`: rule-encoding units of the three sessions; per unit a type II ANOVA of
the square root of hold counts on head angle plus block; the FEV of a factor is its sum of squares
over the sum for both. The A1 unit whose overall F-test gives p > 0.05 is left out (Supplement).

In [ ]:
fev_rows, FULL = [], {}
for ul, u in ENCU[ENCU.session.isin(pfc1.VIDTRACK_SESSIONS)].iterrows():
    j = ANG[u.session]
    counts = pd.Series(np.r_[u.LB_counts, u.PB_counts], index=np.r_[u.LB_trials, u.PB_trials])
    df = j.join(counts.rename("counts"), how="inner").dropna(subset=["counts"])
    df["sqrt_counts"] = np.sqrt(df.counts)
    FULL[ul] = df
    fit = smf.ols("sqrt_counts ~ angl + C(block)", data=df).fit()
    aov = sm.stats.anova_lm(fit, typ=2)
    ssa, ssb = aov.loc["angl", "sum_sq"], aov.loc["C(block)", "sum_sq"]
    fev_rows.append({"ulabel": ul, "region": u.region, "session": u.session, "n_trials": len(df),
                     "fev_block": 100 * ssb / (ssa + ssb), "fev_angl": 100 * ssa / (ssa + ssb), "pF": fit.f_pvalue})
FEV = pd.DataFrame(fev_rows).set_index("ulabel")
print(FEV.round(4).sort_values(["region", "fev_angl"]).to_string())
for reg in ["PFC", "A1"]:
    g = FEV[FEV.region == reg]; gk = g[g.pF <= 0.05]
    print(f"{reg}: units {len(g)}, overall F p > 0.05: {int((g.pF > 0.05).sum())}; after dropping them, median FEV block "
          f"{gk.fev_block.median():.1f}%, head angle {gk.fev_angl.median():.1f}%; head-angle FEV > 50%: {int((gk.fev_angl > 50).sum())}; "
          f"with all units: block {g.fev_block.median():.1f}%, angle {g.fev_angl.median():.1f}%, > 50%: {int((g.fev_angl > 50).sum())}")

**Result: reproduced.** Over 16 mPFC units the median FEV is 94.1% for block and 5.9% for head
angle, and 3 units have head-angle FEV above 50%. In A1 one of 8 units has an overall F-test
p > 0.05 (0.129); without it the medians are 83.0% and 17.0% and one unit is above 50%. All values
match the paper.

### posture-equalized-sign

> On posture-equalized trials the rule encoding persists; its population spread (0.471 in mPFC,
> 0.318 in A1) exceeds that of shuffles (p < 0.005), and only 1 of 16 mPFC and 0 of 8 A1 cells
> switch preferred block (binomial p = 0.0005 and 0.008).

As `3-5-vidtrack/main3.py` to `main5.py`: per session, from the first unit's trials, the
localization trials with the highest head angles and the pitch discrimination trials with the
lowest, the largest equal number for which the localization mean stays above the pitch
discrimination mean (the code's loop stops at the first number where it does not, less one). Rule
encoding is log10 of the pitch discrimination over localization mean count; PRE is the scaled MAD
over units, centered on the median as statsmodels' robust.mad (stand_mad) computes it; the code's
comment says its statsmodels did not subtract the median, so the uncentered MAD is a variant; 10000
shuffles of block labels among the posture-equalized trials, p = (shuffles with a larger PRE + 1) /
10000. A cell switches when the sign of its posture-equalized rule encoding differs from that on all
trials; two-sided binomial test of the switch count.

In [ ]:
rng_pe = np.random.default_rng(3505)
SUBIDX = {}
for ul, df in FULL.items():
    s = ENCU.loc[ul, "session"]
    if s in SUBIDX:
        continue
    lbs = df[df.block == "LB"].sort_values("angl", kind="mergesort"); pbs = df[df.block == "PB"].sort_values("angl", kind="mergesort")
    for n in range(1, len(df)):
        if lbs.angl.to_numpy()[-n:].mean() < pbs.angl.to_numpy()[:n].mean():
            break
    n -= 1
    SUBIDX[s] = {"LB": lbs.index.to_numpy()[-n:], "PB": pbs.index.to_numpy()[:n]}
    print(f"{s}: {n} posture-equalized trials per block; head-angle means LB {df.loc[SUBIDX[s]['LB'], 'angl'].mean():.2f}, "
          f"PB {df.loc[SUBIDX[s]['PB'], 'angl'].mean():.2f} degrees")
pe_rows, FAKE = [], {}
for ul, df in FULL.items():
    u = ENCU.loc[ul]; si = SUBIDX[u.session]
    c = df.counts
    sub_lb, sub_pb = c.loc[si["LB"]], c.loc[si["PB"]]
    rest = df.drop(index=np.r_[si["LB"], si["PB"]])
    pool = np.r_[sub_lb.to_numpy(), sub_pb.to_numpy()]; nl = len(sub_lb)
    perm = np.argsort(rng_pe.random((10000, len(pool))), axis=1)
    fake = pool[perm]
    FAKE[ul] = fake[:, nl:].mean(1) / fake[:, :nl].mean(1)
    pe_rows.append({"ulabel": ul, "region": u.region, "rdiff": u.mPB / u.mLB, "sub_rdiff": sub_pb.mean() / sub_lb.mean(),
                    "nsub_rdiff": rest[rest.block == "PB"].counts.mean() / rest[rest.block == "LB"].counts.mean()})
PE = pd.DataFrame(pe_rows).set_index("ulabel")
PE["switch"] = np.sign(np.log10(PE.sub_rdiff)) != np.sign(np.log10(PE.rdiff))
for reg, exp in [("PFC", 0.471), ("A1", 0.318)]:
    g = PE[PE.region == reg]
    fk = np.log10(np.array([FAKE[u] for u in g.index]))
    for name, f in [("centered (statsmodels robust.mad)", pfc1.stand_mad_centered), ("uncentered (variant)", pfc1.stand_mad_uncentered)]:
        real = f(np.log10(g.sub_rdiff))
        sh = f(fk, axis=0)
        p = (np.sum(sh > real) + 1) / fk.shape[1]
        print(f"{reg} {name}: PRE {real:.3f} (expected {exp} +/- {exp / 10:.4f}), all-trials PRE {f(np.log10(g.rdiff)):.3f}, "
              f"non-equalized PRE {f(np.log10(g.nsub_rdiff)):.3f}; shuffle PRE median {np.median(sh):.3f}, p = {p:.4f}; "
              f"non-finite shuffles {int((~np.isfinite(sh)).sum())}")
    k = int(g.switch.sum())
    print(f"{reg}: {k} of {len(g)} cells switch; two-sided binomial p = {stats.binomtest(k, len(g), 0.5).pvalue:.4f}")

**Result: reproduced.** On the posture-equalized trials (39, 33 and 103 per block in the three
sessions, with block head-angle means within 0.6 degrees), the PRE is 0.471 in mPFC and 0.318 in A1,
the paper's values, and the fraction of shuffles with a larger PRE is 0.0001 and 0.0001 in the two
regions, below 0.005. One of 16 mPFC cells and none of 8 A1 cells switch preferred block, with
binomial p = 0.0005 and 0.0078. The uncentered MAD (variant) gives 0.327 and 0.343.

### posture-equalized-magnitude

> The mean magnitude of rule encoding is not significantly altered on posture-equalized trials,
> in mPFC or A1.

Absolute rule encoding |log10 ratio| on posture-equalized and on the other trials, Wilcoxon
signed-rank per region (M5). The code's own test (`main5.py`), the signed log ratio per region and
preferred block, is a variant.

In [ ]:
for reg in ["PFC", "A1"]:
    g = PE[PE.region == reg]
    a, b = np.abs(np.log10(g.sub_rdiff)), np.abs(np.log10(g.nsub_rdiff))
    print(f"{reg}: mean |log ratio| posture-equalized {a.mean():.3f}, other trials {b.mean():.3f}, change {(a - b).mean():+.3f}; "
          f"signed-rank p = {pfc1.signedrank_r(b, a):.3f} (n = {len(g)})")
    for pref, gg in g.groupby(np.where(g.rdiff > 1, "PB", "LB")):
        print(f"   variant, {pref}-preferring (n = {len(gg)}): signed p = {pfc1.signedrank_r(np.log10(gg.nsub_rdiff), np.log10(gg.sub_rdiff)):.3f}")

**Result: reproduced.** The signed-rank test finds no change in the magnitude of rule encoding,
p = 0.059 in mPFC (16 cells) and 0.141 in A1 (8 cells). The magnitude is smaller on the
posture-equalized trials in both (mean |log10 ratio| 0.308 against 0.418, and 0.218 against 0.401),
so the absence rests on a test of few cells; the code's per-preference variant gives p from 0.09 to
0.36.

## Disruption of mPFC

### fig7-disruption-by-block

> Electrical disruption of mPFC impairs performance; across sessions the impairment is
> significant in pitch discrimination for all three rats and in localization for two of three.

As `7-1-zap/zap.py` and `main1.py` on `miscellaneous/zap/TI_d`: per session, the fraction of hits on
disruption (zap 1) and control (zap 0) trials of block 2 and of block 4, munged trials dropped, the
code's minimum of 5 trials per side; per rat (CR25A Z1, CR28A Z2, CR28B Z3), a two-sided binomial
test of the number of sessions with control above disruption performance. Dropping nonrandom trials
is a variant.

In [ ]:
TI_d = pfc1.load_old_pickle(pfc1.read_member(pfc1.ROOT + "miscellaneous/zap/TI_d"))
ZRAT = {"CR25A": "Z1", "CR28A": "Z2", "CR28B": "Z3"}
print("Sessions per rat:", pd.Series([k[:5] for k in TI_d]).value_counts().sort_index().to_dict())
print("Columns:", list(next(iter(TI_d.values())).columns))
print("zap values:", sorted(set().union(*[set(t.zap.unique()) for t in TI_d.values()])),
      "; fraction of zap trials (munged dropped, non-cue blocks):",
      round(np.mean(np.concatenate([t[(~t.is_munged.astype(bool)) & t.block.isin([2, 4])].zap.to_numpy() for t in TI_d.values()])), 3))

def zap_table(drop_nonrandom):
    rows = []
    for sess, TI in TI_d.items():
        t = TI[~TI.is_munged.astype(bool)]
        t = t[t.outcome.isin(["hit", "error", "wrong_port"])]
        if drop_nonrandom:
            t = t[t.nonrandom == 0]
        hit = (t.outcome == "hit").astype(int)
        for block in [2, 4]:
            for gng in [None, "go", "nogo"]:
                m = (t.block == block) & ((t.go_or_nogo == gng) if gng else True)
                z, c = hit[m & (t.zap == 1)], hit[m & (t.zap == 0)]
                if len(z) < 5 or len(c) < 5:
                    continue
                rows.append({"session": sess, "rat": sess[:5], "block": block, "gng": gng or "all",
                             "zap_m": z.mean(), "ctl_m": c.mean(), "n_zap": len(z), "n_ctl": len(c)})
    return pd.DataFrame(rows)

def rat_tests(Z):
    out = []
    for (rat, block, gng), g in Z.groupby(["rat", "block", "gng"]):
        k = int((g.ctl_m > g.zap_m).sum())
        out.append({"rat": ZRAT[rat], "block": {2: "localization", 4: "pitch discrimination"}[block], "gng": gng,
                    "sessions": len(g), "impaired_sessions": k, "mean_ctl_minus_zap": (g.ctl_m - g.zap_m).mean(),
                    "p": stats.binomtest(k, len(g), 0.5).pvalue})
    return pd.DataFrame(out)

ZT = {dn: zap_table(dn) for dn in [False, True]}
RT7 = {dn: rat_tests(ZT[dn]) for dn in [False, True]}
for dn in [False, True]:
    print(f"\n{'nonrandom trials dropped (variant)' if dn else 'as the code (nonrandom kept)'}:")
    print(RT7[dn][RT7[dn].gng == "all"].round(4).to_string(index=False))

**Result: reproduced.** Control performance exceeds disruption performance in pitch discrimination
in 6 of 6, 8 of 8 and 8 of 8 sessions of Z1, Z2 and Z3 (binomial p = 0.031, 0.0078, 0.0078), and in
localization in 5 of 6 (p = 0.22), 8 of 8 and 8 of 8 (p = 0.0078 each): significant in pitch
discrimination for all three rats and in localization for two. Session counts are 6, 8 and 8.
Dropping nonrandom trials changes nothing.

### fig7c-disruption-nogo

> Disruption impairs nogo trials in particular; every rat is significantly impaired on nogo
> trials in one block or the other, and Z2 also on localization go trials.

As fig7-disruption-by-block, split by go and nogo in each block (`compute_by_block_and_gonogo`).

In [ ]:
for dn in [False, True]:
    r = RT7[dn][RT7[dn].gng != "all"]
    print(f"\n{'nonrandom trials dropped (variant)' if dn else 'as the code (nonrandom kept)'}:")
    print(r.round(4).to_string(index=False))
    nogo_ok = r[(r.gng == "nogo") & (r.p < 0.05)].groupby("rat").size().reindex(["Z1", "Z2", "Z3"]).fillna(0)
    go_sig = r[(r.gng == "go") & (r.p < 0.05)][["rat", "block"]].values.tolist()
    print("rats with a significant nogo impairment in some block:", nogo_ok.to_dict(), "; significant go cases:", go_sig)

**Result: reproduced.** Each rat has a significant nogo impairment in one block (Z1 and Z2 in pitch
discrimination, p = 0.031 and 0.0078; Z3 in localization, p = 0.0078), and the only significant go
impairment is Z2 in localization (8 of 8 sessions, p = 0.0078).

## figs1bc-catch

> Rats perform as well when the distractor is replaced by a novel neutral sound as on standard
> trials, so they select the target and do not memorize the four stimulus pairs.

As `2-1-behavres/main05.py` on `miscellaneous/catch_trial_task`: for each rat and each catch stimulus,
hits and trials on it against the two standard stimuli with the same target pooled; two-sided
Fisher's exact test; all trials of the file, as the code.

In [ ]:
COMPG = [["ri_ca_lc", "ri_lo_lc", "ri_hi_lc"], ["le_ca_lc", "le_lo_lc", "le_hi_lc"],
         ["ca_hi_pc", "le_hi_pc", "ri_hi_pc"], ["ca_lo_pc", "le_lo_pc", "ri_lo_pc"]]
catch_rows = []
for rat in ["CR20B", "CR21A"]:
    TI = pfc1.load_old_pickle(pfc1.read_member(pfc1.ROOT + f"miscellaneous/catch_trial_task/{rat}_trials_info"))
    print(rat, TI.shape, "outcomes", TI.outcome.value_counts().to_dict(), "; catch trials", int(TI.stim_name.str.contains("ca").sum()))
    nh = TI.groupby("stim_name").outcome.apply(lambda x: (x == "hit").sum()); nt = TI.groupby("stim_name").size()
    for g in COMPG:
        s0, n0 = nh[g[0]], nt[g[0]]; s1, n1 = nh[g[1]] + nh[g[2]], nt[g[1]] + nt[g[2]]
        p = stats.fisher_exact([[s0, n0 - s0], [s1, n1 - s1]])[1]
        catch_rows.append({"rat": rat, "catch": g[0], "catch_perf": s0 / n0, "n_catch": n0, "standard_perf": s1 / n1, "n_standard": n1, "p": p})
CATCH = pd.DataFrame(catch_rows)
print(CATCH.round(4).to_string(index=False))
print("Tests with p < 0.05:", int((CATCH.p < 0.05).sum()), "of", len(CATCH))

**Result: reproduced.** None of the eight Fisher's exact tests gives p below 0.05 (smallest 0.11,
CR20B ri_ca_lc, where catch performance is 1.00 against 0.83). CR20B's file holds one session of
390 trials, 51 of them catch trials; CR21A's 663 trials, 99 catch trials.

## fig8-model

> In the network model with 320 neurons, a task signal alone lets the network perform above 80%
> correct with a sensory signal-to-noise ratio as low as 0.0625, and 100% at an SNR of 0.25 for
> virtually any positive task signal.

From the deposited simulation results, as `8-1-model/main1.py`: N = 320, the broad (exponentially
spaced) task-signal gains, the score over 4 x NT_TEST trials, averaged over simulations and the
median taken over repetitions; task 1 (score1b) for positive gains and task 2 (score2b) for the
mirrored negative gains. The code labels the curves SNR = 2^-log2(noise level), that is 1 / noise
level, the gain/noise reading of the criterion (gain/sqrt(noise) does not give the printed grid of
2^-6 to 2^-2).

In [ ]:
t0 = time.time()
resdf = pfc1.load_old_pickle(pfc1.read_member(pfc1.ROOT + "miscellaneous/model/simulation_resdf"))
params = pfc1.load_old_pickle(pfc1.read_member(pfc1.ROOT + "miscellaneous/model/simulation_params.pickle"))
print(f"simulation_resdf: {resdf.shape} rows x columns, read in {time.time() - t0:.0f} s; columns {list(resdf.columns)}")
N_l, noise_l, gains_l, NT = np.asarray(params["N_l"]), np.asarray(params["noise_level_l"]), np.asarray(params["gains_l"]), params["NT_TEST"]
print("N_l", N_l.tolist(), "; noise_level_l", noise_l.tolist(), "; NT_TEST", NT, "; gains", len(gains_l), f"{gains_l.min()} to {gains_l.max()}")
print("SNR (1/noise):", (1 / noise_l).tolist(), "; 1/sqrt(noise):", np.round(1 / np.sqrt(noise_l), 4).tolist())
ggains = np.concatenate([-10 ** (np.linspace(-3, 3, 40))[::-1], 10 ** (np.linspace(-3, 3, 40)), np.linspace(-20, 20, 80)])
assert np.allclose(np.sort(ggains), gains_l)
broad = np.argsort(np.argsort(ggains))[:80]
sub = resdf[(resdf.nN == int(np.where(N_l == 320)[0][0])) & resdf.ngain.isin(broad)]
del resdf
print("score1b, score2b ranges:", sub.score1b.min(), "to", sub.score1b.max(), ",", sub.score2b.min(), "to", sub.score2b.max(), f"(over 4 x {NT} trials)")
piv = sub.groupby(["n_noise_level", "ngain", "nrep"])[["score1b", "score2b"]].mean() / 4.0 / NT
med = piv.groupby(["n_noise_level", "ngain"]).median()
for snr in [0.0625, 0.25]:
    nl = int(np.where(np.isclose(1 / noise_l, snr))[0][0])
    m = med.loc[nl]
    g = gains_l[m.index.to_numpy()]
    pos1 = m.score1b[g > 0]; neg2 = m.score2b[g < 0]
    print(f"SNR {snr}: task 1 at positive gains, max {pos1.max():.3f}, fraction of gains >= 0.99: {np.mean(pos1 >= 0.99):.2f}, >0.8: {np.mean(pos1 > 0.8):.2f}; "
          f"task 2 at negative gains, max {neg2.max():.3f}, fraction >= 0.99: {np.mean(neg2 >= 0.99):.2f}, >0.8: {np.mean(neg2 > 0.8):.2f}")
    x = g / noise_l[nl]
    inplot = (x > 0) & (x <= 10)
    print(f"   within Figure 8B's plotted range (task signal / noise from 0 to 10, 8-1-model/main1.py set_xlim): "
          f"{int(inplot.sum())} gains, fraction >= 0.99: {np.mean(m.score1b[inplot] >= 0.99):.2f}; "
          f"for task signal / noise >= 0.3: {np.mean(m.score1b[(x >= 0.3) & (x <= 10)] >= 0.99):.2f}")
    print("   task 1 by positive gain:", dict(zip(np.round(g[g > 0], 3)[::4], np.round(pos1.to_numpy(), 3)[::4])))
for nl in range(len(noise_l)):
    m = med.loc[nl]; g = gains_l[m.index.to_numpy()]
    print(f"   SNR {1 / noise_l[nl]:.4g}: best task 1 {m.score1b[g > 0].max():.3f}, best task 2 {m.score2b[g < 0].max():.3f}")

**Result: partially reproduced.** With 320 neurons and an SNR of 0.0625 the best positive task
signal gives 93.2% correct in both tasks, above 80%. At an SNR of 0.25 performance reaches 100%,
but it is at least 99% for only 25% of the simulated positive task signals (gains 0.001 to 1000),
and for 30% of those within Figure 8B's plotted range (task signal over noise up to 10): it is
100% for task signal over noise from 0.3 to 10 and lower outside that range (about 0.63 to 0.77
below it, 0.52 at a gain of 84). The first part of the claim holds; the second holds over the
upper part of the plotted range only, not over most positive task signals as the criterion asks.

## Summary

| Claim | Outcome |
|---|---|
| unit-table-link (link) | reproduced |
| fig2a-above-chance | reproduced |
| figs1a-go-better-than-nogo | reproduced |
| fig2d-block-switch | reproduced |
| iti-median | reproduced |
| fig3c-pfc-rule-encoding | reproduced |
| fig4c-a1-rule-encoding | reproduced |
| a1-rule-encoding-not-audresp-dependent | reproduced |
| rule-encoding-effect-size | reproduced |
| fig3ab-pfc-example-units | reproduced |
| fig4ab-a1-example-units | not reproduced (the units named by the code's figure index are swapped; swapped, all values match) |
| fig3d-pfc-interference | reproduced |
| fig4d-a1-interference | reproduced |
| figs2c-s3c-hold-by-trial-type | reproduced |
| figs2f-s3f-bidirectional-vs-spont | reproduced |
| figs2gh-s3gh-rt-mt | reproduced |
| fig5ab-single-unit-onset | reproduced |
| fig5c-population-onset | reproduced |
| fig5d-go-nogo | reproduced |
| audresp-prevalence | reproduced |
| figs5cd-evoked-strength | reproduced |
| figs5e-latency | reproduced |
| fig6cd-baseline-evoked-correlation | reproduced |
| evoked-beyond-baseline-rare | reproduced |
| fig6e-decoding | reproduced |
| figs5fg-tuning-vs-rule | reproduced |
| figs5hi-stimulus-pairs | reproduced |
| figs2i-head-angle | reproduced |
| figs2kl-s3kl-fev | reproduced |
| posture-equalized-sign | reproduced |
| posture-equalized-magnitude | reproduced |
| fig7-disruption-by-block | reproduced |
| fig7c-disruption-nogo | reproduced |
| figs1bc-catch | reproduced |
| fig8-model | partially reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

**How the analysis differs from the paper's.** Every analysis follows the paper's code at commit
390b608 where the claims file names it, ported to Python 3 (R's Mann-Whitney and signed-rank tests
are reproduced with scipy's normal approximations with continuity correction; Benjamini-Hochberg
with statsmodels; logistic regression with scikit-learn's liblinear solver and balanced class
weights). The auditory response class and window come from the code's unit table (M4 reading (a));
recomputing them from the spikes as the Supplement describes (reading (b), a variant) gives 46 mPFC
and 61 A1 responsive units in place of 31 and 49, because the table applies rules the paper does not
state (classes such as "too brief" and "lasts too long"). The single-unit onsets of Figure 5A and 5B
match the paper only with the code's smoother, which moves each spike to the 50 ms sample grid; with
exact spike times the mPFC median is 0.75 s. Bootstrap and shuffle draws use fixed seeds of their
own, so their p-values differ from the paper's draws by sampling error.

**What was found about the data set.** The spike files are pandas 0.10 pickles from Python 2 that
current pandas cannot open, and they hold 2977 clusters of which only the 220 well-sorted units
listed in the code's table were analysed: without that table the units, their quality and their
auditory response windows cannot be identified from the data set. Three sessions have no spike
file (their `klusters0/` holds a README), and two CR21A sessions carry a second spike file outside
`klusters0/` with the same spike times but tetrode groups numbered 0 to 7 and different unit
numbers. The descriptor names the stimulus column `stimulus_name` (it is `stim_name`) and gives a
stimulus numbering that the files do not follow. The code's figure index names the units of
Figures 4A and 4B the other way round from the legends.